**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/laboy-juarez-2019"], check=True)
    os.chdir("/content/encore/entries/laboy-juarez-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Laboy-Juarez et al. (2019) from CRCNS ssc-9

Paper: Laboy-Juarez KJ, Langberg T, Ahn S, Feldman DE (2019). Elementary motion sequence detectors in whisker somatosensory cortex. Nature Neuroscience 22, 1438-1449. doi:10.1038/s41593-019-0448-6.

Dataset: CRCNS ssc-9, "Extracellular recordings in mouse barrel cortex during single- and 2-whisker stimulation", doi:10.6080/K0FJ2F13, read from the public bucket at `https://crcnsarchive.s3-us-west-2.amazonaws.com/ssc-9/live/data.zip`. CRCNS data sets are not versioned (docs/archives.md), so the version is recorded as `unversioned`, pinned by the MD5 sums of the files read, from the data set's `checksums.md5`: `data.zip` 2e7a774017619e76c46db85722587be3 (the only data file; the docs are `crcns_ssc-9_data_description.pdf` d5d1590c38b8eea45e070e8cdfc4827f, version 0.6 of May 13, 2019). There is one published version, the one the paper's Data availability statement points to (crcns.org), so it is the one used; the next cell checks the sum.

Claims file: `claims/laboy-juarez-2019.yaml`, written from the version of record with its supplement. The paper links no code (available on request), so every method below comes from the claims file, the Methods, the legends and the supplement; details taken from the paper cite the passage.

Claims tested and blocked (outcomes are in the summary at the end):

| Claim | Status |
|---|---|
| exp1-units to supp2-sw-tuned-sharpest-cwsw (experiment 1, 10 claims) | tested |
| exp2-units to supp8-order-decoding (experiment 2, 31 claims) | tested |
| exp3-units-cw-tuned to fig8k-awake-binary-decoder (experiment 3, 8 claims) | tested |
| fig8b-awake-behavior | blocked: the files hold no lick times or trial types |

In [ ]:
import hashlib
import fsspec
import lj2019 as L
h = hashlib.md5()
with fsspec.open(L.URL, block_size=2 ** 22) as f:  # streamed in chunks, not stored
    for chunk in iter(lambda: f.read(2 ** 22), b""):
        h.update(chunk)
print(f"CRCNS {L.DATASET}, version {L.VERSION}: data.zip MD5 {h.hexdigest()}, matches checksums.md5: {h.hexdigest() == L.DATA_MD5}")

In [ ]:
import resource
import time

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import lj2019 as L

t0 = time.time()
z = L.open_zip()
U1, T1, P1 = L.load_experiment(z, 1)
U3, T3, P3 = L.load_experiment(z, 3)
U2, T2, P2 = L.load_experiment(z, 2)
print(f"loaded in {time.time() - t0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check

What was loaded, by experiment, recording and condition, with the checks of the data descriptor against the values.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
for e, (U, T) in {1: (U1, T1), 2: (U2, T2), 3: (U3, T3)}.items():
    print(f"=== Experiment {e}: {U.file.nunique()} files, {len(T)} recordings (distinct trial sequences), {len(U)} units")
    rows = []
    for rec, tab in T.items():
        kc = tab.kind.value_counts()
        rows.append(dict(recording=rec, units=int((U.recording == rec).sum()), trials=len(tab),
                         single=kc.get("single", 0), cwsw=kc.get("cwsw", 0), swsw=kc.get("swsw", 0),
                         triple=kc.get("triple", 0), sham=kc.get("sham", 0),
                         dt_values=tab.loc[tab.kind == "cwsw", "dt"].nunique(),
                         dt_min=tab.loc[tab.kind == "cwsw", "dt"].min(), dt_max=tab.loc[tab.kind == "cwsw", "dt"].max(),
                         trial_ms=L.trial_length(U, rec) if e != 3 else np.nan))
    print(pd.DataFrame(rows).to_string(index=False))
    print("units per file:", U.groupby("file").size().to_dict())
    dup = U[U.spike_hash.duplicated(keep=False)]
    print("units with identical spike times (duplicates):", len(dup))
    if len(dup):
        print(dup[["unit", "file", "recName", "depth", "n_spikes"]].to_string(index=False))
    mism = U.groupby("recording").recName.apply(lambda s: sorted(set(x.rsplit("_ch", 1)[0] for x in s)))
    mism = mism[mism.apply(len) > 1]
    print("recordings whose units carry more than one recName stem:", mism.to_dict())
    print("depth range (um):", U.depth.min(), "to", U.depth.max())

In [ ]:
print("Stimulus directions (1 caudal, -1 rostral, data descriptor) by experiment:")
for e, T in [(1, T1), (2, T2), (3, T3)]:
    allt = pd.concat([t[["kind", "a", "b", "dt", "da", "db"]] for t in T.values()])
    s = allt[allt.kind == "single"]
    p = allt[allt.kind.isin(["cwsw", "swsw"])]
    print(f"  exp {e}: single-whisker trials by direction {s.da.value_counts().to_dict()}; two-whisker trials by (first, second) direction "
          f"{pd.Series(list(zip(p.da, p.db))).value_counts().to_dict()}")
t510 = [t for r, t in T1.items() if r.startswith("20170510")]
for t in t510:
    p = t[t.kind.isin(["cwsw", "swsw"])]
    mixed = p[p.da != p.db]
    rostral = sorted(set(mixed.loc[mixed.da == -1, "a"]) | set(mixed.loc[mixed.db == -1, "b"]))
    print(f"  20170510 recording: {len(mixed)} of {len(p)} two-whisker trials have opposite directions; the whisker deflected rostrally in them is one of",
          rostral, "; same-direction trials of the pairs that involve 3 or 8:",
          int(((p.da == p.db) & (p.a.isin([3, 8]) | p.b.isin([3, 8]))).sum()))
print("Three-whisker stimuli (not in the paper or the descriptor):")
for r, t in T2.items():
    if (t.kind == "triple").any():
        print(f"  {r}: {(t.kind == 'triple').sum()} trials")
cw2 = pd.concat([t.loc[t.kind == "cwsw", ["b", "dt", "da", "db"]].assign(rec=r) for r, t in T2.items()])
print("Experiment 2 CW-SW dt values:", int(cw2.dt.min()), "to", int(cw2.dt.max()), "in", cw2.dt.nunique(), "values (the paper says 100 values within +-50 ms)")
same = cw2.da == cw2.db
print("Experiment 2 trials per CW-SW combination and dt, mean over recordings: correlated",
      round(cw2[same].groupby(["rec", "b", "dt"]).size().mean(), 2), "| anti-correlated", round(cw2[~same].groupby(["rec", "b", "dt"]).size().mean(), 2),
      "(the paper gives 8.42 +- 4.2 per unique sequence)")
print("Experiment 3 epochs: spikes after 300 ms, by file:")
for f, g in U3.groupby("file"):
    print(f"  {f}: {g.n_out.sum()} of {g.n_spikes.sum()} spikes ({g.n_out.sum() / g.n_spikes.sum():.2f}), largest time {g.max_t.max():.0f} ms")
import re
ocols = lambda t, p: [c for c in t.columns if re.fullmatch(p + r"\d+", c)]
late = np.concatenate([(t[ocols(t, "o")].to_numpy() > 0).ravel() for t in T3.values()])
print(f"  epochs (unit x epoch) holding any spike after 300 ms: {late.mean():.3f}")
inwin = np.concatenate([((t[ocols(t, "o")].to_numpy() > 0) & (t[ocols(t, "c")].to_numpy() > 0)).ravel() for t in T3.values()])
print(f"  of these, with a spike in 0-125 ms as well: {inwin.sum() / max(late.sum(), 1):.3f}")
print("Experiment 3 experiment field: whiskers by recording", U3.groupby("recording").whiskers.first().to_dict())
print("Experiment 3 tip depth (um):", U3.groupby("recording").depth.first().to_dict(), "; no per-unit depth in the files")

The raw fields behind the direction check: the fields each unit's struct holds in experiments 1 and 3, the raw stimulus fields of the first two-whisker trials with opposite directions in one 20170510 recording, and the trials per pair and direction pattern there.

In [ ]:
import io
import scipy.io as sio
raw1 = np.atleast_1d(sio.loadmat(io.BytesIO(z.read("Experiment1/20170510.mat")), squeeze_me=True, struct_as_record=False)["UnitInfoExp1"])
raw3 = np.atleast_1d(sio.loadmat(io.BytesIO(z.read("Experiment3/20181114.mat")), squeeze_me=True, struct_as_record=False)["UnitInfoExp3"])
print("experiment 1 unit fields:", list(raw1[0]._fieldnames))
print("experiment 3 unit fields:", list(raw3[0]._fieldnames), "; fields of its 'experiment' struct:", list(raw3[0].experiment._fieldnames))
u = raw1[0]
mixed = [i for i, r in enumerate(u.directionWs) if np.size(r) == 2 and len(set(L._vec(r))) == 2][:6]
print(f"20170510, unit {u.recName}: raw fields of the first six two-whisker trials with opposite directions")
for i in mixed:
    print(f"  trial {i}: DeflectedWs {L._vec(u.DeflectedWs[i])}, timeWs {L._vec(u.timeWs[i])}, directionWs {L._vec(u.directionWs[i])}")
tab = L._trial_table_12(u)
p = tab[tab.kind.isin(["cwsw", "swsw"])]
cnt = p.assign(pattern=np.where(p.da == p.db, "same", "opposite")).groupby(["a", "b", "pattern"]).size().unstack(fill_value=0)
has38 = cnt.index.get_level_values(0).isin([3, 8]) | cnt.index.get_level_values(1).isin([3, 8])
print("trials per pair (first whisker, second whisker) by direction pattern, the 15 pairs with whisker 3 or 8:")
print(cnt[has38].T.to_string())
print(f"the other {int((~has38).sum())} pairs: same-direction trials per pair {cnt.loc[~has38, 'same'].min()} to {cnt.loc[~has38, 'same'].max()}, "
      f"opposite-direction trials in total {int(cnt.loc[~has38, 'opposite'].sum())}")
print("direction patterns (first, second) of the opposite trials:", p[p.da != p.db].groupby(["da", "db"]).size().to_dict())
del raw1, raw3, u

The experiment 3 files give no deflection times. The latency of the CW response in CW-SW epochs shows how DeltaT and the epoch clock are defined: if epoch time 0 is the first deflection and dt = t_SW - t_CW, the CW-driven response comes 40 ms later when the SW leads (dt = -40) than when the CW leads (dt = +40). Mean counts over the 81 units in 10 ms bins.

In [ ]:
bins = np.arange(-50, 200, 10)
def mean_psth(cond):
    tot, n = 0, 0
    for rec, tab in T3.items():
        P = P3[rec]
        m = cond(tab).to_numpy()[P["rows"]]
        for k in U3.loc[U3.recording == rec, "unit"]:
            tot = tot + P[f"h{k}"][m].sum(0)
            n += m.sum()
    return tot / n
for lab, cond in [("CW alone", lambda t: (t.kind == "single") & (t.a == L.CW3)),
                  ("CW-SW, dt = +40 (CW at 0)", lambda t: (t.kind == "cwsw") & (t.dt == 40)),
                  ("CW-SW, dt = -40 (SW at 0, CW at 40)", lambda t: (t.kind == "cwsw") & (t.dt == -40)),
                  ("SW alone", lambda t: (t.kind == "single") & (t.a != L.CW3)),
                  ("sham", lambda t: t.kind == "sham")]:
    h = mean_psth(cond)
    print(f"{lab:38s} peak bin starts at {bins[h.argmax()]:4d} ms; per-bin means x100:", (100 * h).round(1)[3:16])
print("bins shown start at", bins[3:16])

Clock check for every recording: the nine single whiskers are decoded from the population's counts in windows before and after the deflection onset (multinomial logistic regression, fivefold cross-validation; chance 1/9). A shared clock gives chance before onset and above chance after it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
wins = {"-50 to 0 ms": slice(0, 5), "0 to 50 ms": slice(5, 10), "50 to 100 ms": slice(10, 15)}
rows = []
for e, (U, T, P) in {1: (U1, T1, P1), 2: (U2, T2, P2), 3: (U3, T3, P3)}.items():
    for rec, tab in T.items():
        Pm = P[rec]
        sub = tab[Pm["rows"]]
        m = (sub.kind == "single").to_numpy() & ((sub.da == 1).to_numpy())
        ks = U.loc[U.recording == rec, "unit"].tolist()
        r = dict(exp=e, recording=rec, units=len(ks), trials=int(m.sum()))
        for w, sl in wins.items():
            X = np.stack([Pm[f"h{k}"][m][:, sl].sum(1) for k in ks], 1).astype(float)
            r[w] = cross_val_score(LogisticRegression(max_iter=300), X, sub.a.to_numpy()[m], cv=5).mean()
        rows.append(r)
CLK = pd.DataFrame(rows)
print(CLK.round(3).to_string(index=False))
print("mean accuracy by window:", CLK[list(wins)].mean().round(3).to_dict(), "; chance", round(1 / 9, 3))

## Experiment 1

Per unit: the mean count in the 125 ms window for each single whisker deflected caudally (the paper's nine single-whisker stimuli; the files also hold rostral deflections, see the data check), for each of the 252 sequences (36 combinations x 7 dt) over its same-direction trials, and the spontaneous rate from sham trials under both sham readings (O1). Same-direction trials are all caudal-caudal here. The Methods list experiment 1's 261 stimuli as 9 single-whisker deflections and 252 two-whisker sequences (Methods, "Whisker stimulation"), and say the experiment 3 stimuli were "identical to experiment 1" apart from amplitude and dt and that "Only same-direction (correlated) stimuli were presented" (Methods, "Recordings in awake behaving mice"). The two 20170510 recordings also hold opposite-direction trials of the pairs that involve whisker 3 or 8 (data check); they are left out of the main analysis and pooled in as a variant.

In [ ]:
ALLDIRS = ((1, 1), (1, -1), (-1, 1), (-1, -1))
def exp1_table(U, T, single_dirs=(1,), pair_dirs=ALLDIRS):
    rows, seqs = [], {}
    for rec, tab in T.items():
        ks = U.loc[U.recording == rec, "unit"].tolist()
        length = L.trial_length(U, rec)
        cols = [f"c{k}" for k in ks]
        sing = tab[(tab.kind == "single") & tab.da.isin(single_dirs)].groupby("a")[cols].mean()
        pd_ok = pd.Series(list(zip(tab.da, tab.db)), index=tab.index).isin(pair_dirs)
        pair = tab[tab.kind.isin(["cwsw", "swsw"]) & pd_ok].groupby(["kind", "a", "b", "dt"])[cols].mean()
        for k in ks:
            r = dict(unit=k, recording=rec)
            for sr in "ab":
                G, p = L.lrt_responsive(tab, k, (tab.kind == "single") & tab.da.isin(single_dirs), sr, length)
                r[f"p_resp_{sr}"] = p
                r[f"spont_{sr}"] = L.spont_rate(tab, k, sr, length)
            for w in range(1, 10):
                r[f"s{w}"] = sing.loc[w, f"c{k}"]
            rows.append(r)
            seqs[k] = pair[f"c{k}"]
    df = U[["unit", "file", "recording", "depth", "animal"]].merge(pd.DataFrame(rows), on=["unit", "recording"])
    df["layer"] = df.depth.apply(L.layer_of)
    return df, seqs

SAME = ((1, 1), (-1, -1))
E1, SEQ1 = exp1_table(U1, T1, pair_dirs=SAME)
E1all, SEQ1all = exp1_table(U1, T1, single_dirs=(1, -1), pair_dirs=SAME)
print(E1.groupby("layer").size().to_dict())

### exp1-units

> In experiment 1, 142 of 160 single units from 8 recording sites in 5 mice were whisker-responsive, 36 in L2/3, 48 in L4, 44 in L5 and 14 in L6.

In [ ]:
exp = {"units": 160, "responsive": 142, "L2/3": 36, "L4": 48, "L5": 44, "L6": 14}
out = []
for name, df in [("caudal (main)", E1), ("both directions pooled", E1all)]:
    for sr in "ab":
        resp = df[df[f"p_resp_{sr}"] < 0.05]
        row = dict(singles=name, sham_reading=sr, units=len(df), recordings=df.recording.nunique(),
                   mice=df.animal.nunique(), responsive=len(resp))
        for lay in ["L2/3", "L4", "L5", "L6", "outside"]:
            row[lay] = int((resp.layer == lay).sum())
        row["all within 10%"] = all(abs(row[q] - v) <= 0.1 * v for q, v in exp.items() if q != "units") and abs(len(df) - 160) <= 16
        out.append(row)
print(pd.DataFrame(out).to_string(index=False))
print("all units by layer:", E1.layer.value_counts().to_dict())
print("responsive units per mouse (caudal, sham reading a):",
      E1[E1.p_resp_a < 0.05].groupby("animal").size().to_dict(), "of", E1.groupby("animal").size().to_dict())

In [ ]:
R1 = E1[E1.p_resp_a < 0.05].copy()  # responsive units, sham reading (a); see exp1-units
W = [f"s{w}" for w in range(1, 10)]
R1["best_w"] = R1[W].to_numpy().argmax(axis=1) + 1
R1["cw_tuned"] = R1.best_w == L.CW12
print(len(R1), "responsive units used below (sham reading a); CW-tuned", int(R1.cw_tuned.sum()))

### fig1b-somatotopy

> On average across experiment 1 units, the CW evoked the most spikes, followed by the within-row SWs.

In [ ]:
pos = {w: L.POSNAME[L.relpos(w, 1)] for w in range(1, 10)}
m = R1[W].mean()
sem = R1[W].sem()
tab_ = pd.DataFrame({"position": [pos[w] for w in range(1, 10)], "mean": m.values, "sem": sem.values}, index=range(1, 10)).sort_values("mean", ascending=False)
print(tab_.round(3).to_string())
order = tab_.position.tolist()
print("CW largest:", order[0] == "CW", "| next two:", order[1:3])
print("by layer:")
print(R1.groupby("layer")[W].mean().rename(columns={f"s{w}": pos[w] for w in range(1, 10)}).round(2).T.to_string())

**Result: reproduced.** The CW has the largest mean response over the 144 responsive units (0.517 spikes per stimulus), followed by the rostral (0.368) and caudal (0.327) within-row SWs. The same order holds under the three variants of the experiment 1 section below.

### exp1-cw-tuned-fraction

> In experiment 1, 61% of responsive units were driven most strongly by the CW (89 units) and 39% by a SW (53 units).

In [ ]:
for sr in "ab":
    r = E1[E1[f"p_resp_{sr}"] < 0.05]
    cw = int((r[W].to_numpy().argmax(axis=1) + 1 == L.CW12).sum())
    print(f"sham reading {sr}: responsive {len(r)}, CW-tuned {cw}, SW-tuned {len(r) - cw}, fraction CW {cw / len(r):.3f}")
print("by mouse (reading a):", R1.groupby("animal").cw_tuned.agg(["sum", "size"]).to_dict("index"))

**Result: reproduced (resting on an assumption).** 87 of 144 responsive units are CW-tuned and 57 SW-tuned (fraction 0.604), all within tolerance of 89, 53 and 0.61; sham reading (b) gives 89 and 58 (0.605). With both directions of singles pooled, 86 are CW-tuned and 69 SW-tuned, outside the tolerance of 53, so the outcome rests on the caudal reading.

### exp1-cw-sw-comparable-firing

> CW-tuned and SW-tuned units had comparable spontaneous activity (1.28 +- 0.24 and 2.08 +- 0.4 Hz) and maximal evoked firing (4.8 +- 0.48 and 4.56 +- 0.64 spikes per stimulus).

In [ ]:
R1["max_single"] = R1[W].max(axis=1)
R1["max_all"] = [max(R1.loc[i, "max_single"], SEQ1[k].max()) for i, k in zip(R1.index, R1.unit)]
rows = []
for col, lab in [("spont_a", "spont, sham reading a (Hz)"), ("spont_b", "spont, sham reading b (Hz)"),
                 ("max_single", "max evoked, reading a: best single"), ("max_all", "max evoked, reading b: best of 261")]:
    a, b = R1.loc[R1.cw_tuned, col], R1.loc[~R1.cw_tuned, col]
    t = stats.ttest_ind(a, b)
    rows.append(dict(measure=lab, cw_mean=a.mean(), cw_sem=a.sem(), sw_mean=b.mean(), sw_sem=b.sem(), t=t.statistic, p=t.pvalue))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("variant, not the claims file's unit: maximal evoked firing as a rate in Hz over the 125 ms window (count / 0.125 s)")
for col, lab in [("max_single", "reading a, best single"), ("max_all", "reading b, best of 261")]:
    a, b = R1.loc[R1.cw_tuned, col] / (L.WINDOW / 1000), R1.loc[~R1.cw_tuned, col] / (L.WINDOW / 1000)
    print(f"   {lab}: CW-tuned {a.mean():.2f} +- {a.sem():.2f} Hz, SW-tuned {b.mean():.2f} +- {b.sem():.2f} Hz; "
          f"within 4.8 +- 0.48 and 4.56 +- 0.64: {abs(a.mean() - 4.8) <= 0.48 and abs(b.mean() - 4.56) <= 0.64}")
print("expected: spont 1.28 +- 0.24 (CW), 2.08 +- 0.4 (SW); max evoked 4.8 +- 0.48 (CW), 4.56 +- 0.64 (SW)")

**Result: partially reproduced.** Spontaneous rates match: 1.33 +- 0.14 Hz for CW-tuned and 2.05 +- 0.43 Hz for SW-tuned units under sham reading (a) (1.27 and 2.01 under (b)), within the paper's 1.28 +- 0.24 and 2.08 +- 0.4. Maximal evoked firing in spikes per stimulus, the unit of the claims file, does not: the best single-whisker response is 0.58 and 0.60 and the best response over all 261 stimuli 0.86 and 0.91, against 4.8 and 4.56. CW-tuned and SW-tuned units do not differ in maximal firing (p = 0.84 and 0.65); the spontaneous difference gives p = 0.067 and 0.042. Read as a rate in Hz over the 125 ms window, the best single-whisker response is 4.64 +- 0.42 and 4.80 +- 0.72 Hz, within the paper's values; this reading is a question for a reader (questions file) and does not decide the outcome.

### supp1d-sw-tuning-broader

> Single-whisker tuning was broader for SW-tuned than for CW-tuned units in experiment 1 (one-factor ANOVA on ranked tuning curves, p = .0003).

In [ ]:
X = R1[W].to_numpy()
ranked = -np.sort(-X, axis=1) / X.max(axis=1, keepdims=True)
cwm = R1.cw_tuned.to_numpy()
a_cw, a_sw = ranked[cwm, 1:].mean(axis=1), ranked[~cwm, 1:].mean(axis=1)
fa = stats.f_oneway(a_cw, a_sw)
print(f"(a) mean of ranks 2-9: CW-tuned {a_cw.mean():.3f}, SW-tuned {a_sw.mean():.3f}, F = {fa.statistic:.2f}, p = {fa.pvalue:.2g}")
fb = stats.f_oneway(ranked[cwm, 1:].ravel(), ranked[~cwm, 1:].ravel())
print(f"(b) pooled rank values (ranks 2-9): CW {ranked[cwm, 1:].mean():.3f}, SW {ranked[~cwm, 1:].mean():.3f}, F = {fb.statistic:.2f}, p = {fb.pvalue:.2g}")
fb9 = stats.f_oneway(ranked[cwm].ravel(), ranked[~cwm].ravel())
print(f"(b') pooled rank values with rank 1: p = {fb9.pvalue:.2g}")
print("mean ranked curve CW-tuned:", ranked[cwm].mean(0).round(2), "\nmean ranked curve SW-tuned:", ranked[~cwm].mean(0).round(2))

**Result: reproduced.** The mean normalized response over ranks 2 to 9 is 0.482 for SW-tuned and 0.393 for CW-tuned units, a one-factor ANOVA gives p = 0.00072 under reading (a) and p = 1.6e-10 under reading (b).

### fig1d-best-sequence-exceeds-single

> Nearly all experiment 1 units spiked more to their best two-whisker combination than to their best single whisker, with a slope of 1.26.

In [ ]:
R1["best_seq"] = [SEQ1[k].max() for k in R1.unit]
x, y = R1.max_single.to_numpy(), R1.best_seq.to_numpy()
slope0 = (x @ y) / (x @ x)
lr = stats.linregress(x, y)
print(f"(a) slope through origin {slope0:.3f}; (b) slope with intercept {lr.slope:.3f} (intercept {lr.intercept:.3f})")
print(f"units with best sequence > best single: {np.sum(y > x)} of {len(x)} ({np.mean(y > x):.3f})")
print("ranges: best single", x.min().round(2), "-", x.max().round(2), "; best sequence", y.min().round(2), "-", y.max().round(2))
print("by mouse, fraction above:", R1.assign(above=y > x).groupby("animal").above.mean().round(2).to_dict())

**Result: reproduced.** The slope of best sequence on best single whisker is 1.258 with an intercept (reading b, within 1.26 +- 0.126) and 1.379 through the origin (reading a, outside); 143 of 144 units (0.993) respond more to their best sequence.

### fig1e-best-combination-includes-cw

> The best combination included the CW for 70% of all units, 85% of CW-tuned units and 47% of SW-tuned units, against a chance level of 22%.

In [ ]:
def best_combo(s):
    comb = s.groupby(level=["kind", "a", "b"]).max()
    return comb.idxmax()
bc = [best_combo(SEQ1[k]) for k in R1.unit]
R1["best_has_cw"] = [b[0] == "cwsw" for b in bc]
print(f"all {R1.best_has_cw.mean():.3f} (n={len(R1)}), CW-tuned {R1.loc[R1.cw_tuned, 'best_has_cw'].mean():.3f} (n={R1.cw_tuned.sum()}), "
      f"SW-tuned {R1.loc[~R1.cw_tuned, 'best_has_cw'].mean():.3f} (n={(~R1.cw_tuned).sum()}); chance 8/36 = {8 / 36:.3f}")
print("by mouse:", R1.groupby("animal").best_has_cw.mean().round(2).to_dict())

**Result: reproduced (resting on an assumption).** With same-direction sequence trials, the best combination contains the CW for 0.694 of all units, 0.851 of CW-tuned units and 0.456 of SW-tuned units, all within tolerance; chance is 0.222. In the two 20170510 recordings the pair of whiskers 3 and 8 was delivered only with opposite directions (data check), so their 38 units have 35 combinations. Pooling the opposite-direction trials in (variant ii below) gives 0.688, 0.862 and 0.421, the last just below the tolerance band (0.423 to 0.517); both directions of singles pooled (variant i) gives 0.677, 0.826 and 0.493, within tolerance.

### fig1f-cwsw-exceeds-swsw

> The best CW-SW sequence evoked more spikes than the best SW-SW sequence.

In [ ]:
rows = []
for reading in ["a", "b"]:
    d = []
    for i, k in zip(R1.index, R1.unit):
        s = SEQ1[k]
        comb = s.groupby(level=["kind", "a", "b"]).max()
        allr = np.r_[R1.loc[i, W].to_numpy(float), s.to_numpy()] if reading == "a" else comb.to_numpy()
        mu, sd = allr.mean(), allr.std(ddof=1)
        d.append(((comb["cwsw"].max() - mu) / sd, (comb["swsw"].max() - mu) / sd))
    d = np.array(d)
    t = stats.ttest_rel(d[:, 0], d[:, 1])
    rows.append(dict(reading=reading, above_diagonal=int(np.sum(d[:, 0] > d[:, 1])), n=len(d),
                     mean_diff=(d[:, 0] - d[:, 1]).mean(), t=t.statistic, p=t.pvalue))
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** 100 of 144 units lie above the diagonal under both z-scoring readings, with a mean z-scored difference of 0.61 (reading a, paired t-test p = 6e-8) and 0.54 (reading b, p = 4e-8).

### fig1g-cwsw-sharper and supp2-sw-tuned-sharpest-cwsw

> Neurons were more sharply tuned, by lifetime sparseness, among CW-SW sequences than among SW-SW sequences.

> SW-tuned units had particularly sharp tuning for CW-SW sequences.

In [ ]:
Ns = np.arange(2, 57)
def topN_sparse(v, Ns):
    v = -np.sort(-np.asarray(v, float))
    return np.array([L.sparseness(v[:n]) for n in Ns])
SPc = np.array([topN_sparse(SEQ1[k]["cwsw"].to_numpy(), Ns) for k in R1.unit])
SPs = np.array([topN_sparse(SEQ1[k]["swsw"].to_numpy(), Ns) for k in R1.unit])
print("sequences per unit: CW-SW", len(SEQ1[R1.unit.iloc[0]]["cwsw"]), "SW-SW", len(SEQ1[R1.unit.iloc[0]]["swsw"]))
mc, ms = SPc.mean(0), SPs.mean(0)
print("mean sparseness at N = 2, 5, 10, 20, 40, 56: CW-SW", mc[[0, 3, 8, 18, 38, 54]].round(3), "SW-SW", ms[[0, 3, 8, 18, 38, 54]].round(3))
print(f"CW-SW above SW-SW at {np.sum(mc > ms)} of {len(Ns)} values of N")
print("paired t-test at each N (descriptive), largest p:", max(stats.ttest_rel(SPc[:, i], SPs[:, i]).pvalue for i in range(len(Ns))))

**Result: reproduced.** The mean lifetime sparseness of the N best CW-SW sequences is above that of the N best SW-SW sequences at all 55 values of N from 2 to 56 (for example 0.097 against 0.046 at N = 56). The 20170510 units have 189 SW-SW sequences, as the pair 3-8 has no same-direction trials there.

In [ ]:
cwm = R1.cw_tuned.to_numpy()
msw, mcw = SPc[~cwm].mean(0), SPc[cwm].mean(0)
print("CW-SW sparseness, SW-tuned at N = 2, 5, 10, 20, 40, 56:", msw[[0, 3, 8, 18, 38, 54]].round(3))
print("CW-SW sparseness, CW-tuned at N = 2, 5, 10, 20, 40, 56:", mcw[[0, 3, 8, 18, 38, 54]].round(3))
print(f"SW-tuned above CW-tuned at {np.sum(msw > mcw)} of {len(Ns)} values of N")

**Result: reproduced.** The CW-SW sparseness of SW-tuned units is above that of CW-tuned units at all 55 values of N (0.136 against 0.072 at N = 56).

### Experiment 1 under the other readings of the stimulus directions and of the sham window

The main analysis above uses the caudal single-whisker deflections, same-direction sequence trials and sham reading (a). The same measures with (i) both directions of the single-whisker deflections pooled, (ii) every sequence trial, including the opposite-direction trials of the 20170510 recordings, and (iii) sham reading (b) for responsiveness and spontaneous rates (147 responsive units):

In [ ]:
def exp1_summary(E, SEQ, sr="a"):
    R = E[E[f"p_resp_{sr}"] < 0.05].copy()
    R["cw_tuned"] = R[W].to_numpy().argmax(1) + 1 == L.CW12
    x = R[W].max(1).to_numpy(); y = np.array([SEQ[k].max() for k in R.unit])
    lr_ = stats.linregress(x, y)
    bc_ = np.array([best_combo(SEQ[k])[0] == "cwsw" for k in R.unit])
    m_ = R[W].mean()
    order_ = [L.POSNAME[L.relpos(int(c[1:]), 1)] for c in m_.sort_values(ascending=False).index]
    Xr = R[W].to_numpy(); rk = -np.sort(-Xr, 1) / Xr.max(1, keepdims=True)
    cwm_ = R.cw_tuned.to_numpy()
    return dict(responsive=len(R), cw_tuned=int(cwm_.sum()), sw_tuned=int((~cwm_).sum()),
                spont_cw=R.loc[cwm_, f"spont_{sr}"].mean(), spont_sw=R.loc[~cwm_, f"spont_{sr}"].mean(),
                max_single_cw=R.loc[cwm_, W].max(1).mean(), max_single_sw=R.loc[~cwm_, W].max(1).mean(),
                p_broader=stats.f_oneway(rk[cwm_, 1:].mean(1), rk[~cwm_, 1:].mean(1)).pvalue,
                slope_intercept=lr_.slope, frac_above=np.mean(y > x),
                best_cw_all=bc_.mean(), best_cw_cwtuned=bc_[cwm_].mean(), best_cw_swtuned=bc_[~cwm_].mean(),
                order=" > ".join(order_[:3]),
                **{f"{lay}": int((R.layer == lay).sum()) for lay in ["L2/3", "L4", "L5", "L6"]})
_, SEQ1alltr = exp1_table(U1, T1)
var1 = pd.DataFrame({"main (caudal singles, same-direction sequences, sham a)": exp1_summary(E1, SEQ1),
                     "(i) both directions of singles": exp1_summary(E1all, SEQ1all),
                     "(ii) every sequence trial": exp1_summary(E1, SEQ1alltr),
                     "(iii) sham reading b": exp1_summary(E1, SEQ1, "b")})
print(var1.to_string())

**Result: reproduced (resting on an assumption).** With caudal single-whisker deflections and sham reading (a), 144 of the 160 units in 8 recordings from 5 mice are responsive, 36 in L2/3, 48 in L4, 46 in L5 and 14 in L6, every count within its tolerance; sham reading (b) gives 147 with 36, 50, 47 and 14. Pooling both deflection directions of the singles gives 155 responsive units with 52 in L5, outside its tolerance (44 +- 4.4), so the outcome rests on reading the paper's nine single-whisker stimuli as the caudal ones. In the table above, the sequence measures (slope, best combination) under sham reading (b) are 1.254 and 0.694, 0.843 and 0.466, all within the tolerances of fig1d and fig1e; with every sequence trial (ii) the SW-tuned fraction of fig1e falls to 0.421.

## Experiment 2: responses, dt smoothing and the shuffle test

For each unit and each stimulus set (correlated, anti-correlated): responsiveness by the Poisson test under the direction readings (O2) and sham readings (O1); the eight CW-SW dt tuning curves on the 1 ms grid, smoothed by the cross-validated Gaussian method of the Methods; the cwSTRF, best combination and best dt; the CSI trace; and the shuffle test (5000 shuffles). The duplicate unit found in the data check (unit 179, the same spike train as unit 178) is dropped.

In [ ]:
t0 = time.time()
KER = L.gaussian_kernels(101, L.SDS)
rng_sm = np.random.default_rng(20190801)
dup2 = set(U2.loc[U2.spike_hash.duplicated(keep="first"), "unit"])
rows2, CUR = [], {}
for rec, tab in T2.items():
    length = L.trial_length(U2, rec)
    ks = [k for k in U2.loc[U2.recording == rec, "unit"] if k not in dup2]
    sing = tab[tab.kind == "single"]
    for which in ["corr", "anti"]:
        dirs = L.set_dirs(tab, which)
        smask_a = L.single_mask_for(tab, dirs)
        smask_b = (tab.kind == "single").to_numpy()
        st = tab[L.set_mask(tab, which)]
        comb = st.b.map({w: i for i, w in enumerate(L.SWS12)}).to_numpy()
        dti = (st.dt.to_numpy() + 50).astype(int)
        # direction pairs of each trial of the set, for the direction-specific linear prediction
        for k in ks:
            r = dict(unit=k, recording=rec, set=which)
            for o2, sm_ in [("a", smask_a), ("b", smask_b)]:
                for sr in "ab":
                    r[f"p_{o2}{sr}"] = L.lrt_responsive(tab, k, sm_, sr, length)[1]
            r["spont_a"] = L.spont_rate(tab, k, "a", length)
            r["spont_b"] = L.spont_rate(tab, k, "b", length)
            means = sing.groupby(["a", "da"])[f"c{k}"].mean()
            vars_ = sing.groupby(["a", "da"])[f"c{k}"].var()
            ns_ = sing.groupby(["a", "da"])[f"c{k}"].size()
            y = st[f"c{k}"].to_numpy(float)
            # O2 (a): linear prediction per trial from its own directions, averaged per combination
            md = means.to_dict()
            lin_a = np.array([np.mean([md[(L.CW12, da)] + md[(w, db)] for da, db in zip(st.da[st.b == w], st.db[st.b == w])])
                              for w in L.SWS12])
            mb = sing.groupby("a")[f"c{k}"].mean()
            lin_b = np.array([mb.loc[L.CW12] + mb.loc[w] for w in L.SWS12])
            curves, status, kidx, lls = np.zeros((8, 101)), [], [], []
            for ci in range(8):
                sel = comb == ci
                cv, ki, stt, raw, ll = L.fast_dt_smooth(y[sel], dti[sel], rng_sm, KER, return_ll=True)
                curves[ci] = cv
                status.append(stt)
                kidx.append(ki)
                lls.append(ll)
            CUR[(k, which)] = dict(curves=curves, status=status, kidx=kidx, ll=lls, y=y, comb=comb, dti=dti,
                                  lin_a=lin_a, lin_b=lin_b, single_means=means, single_vars=vars_, single_n=ns_,
                                  dirs=dirs)
            rows2.append(r)
    print(rec, f"{time.time() - t0:.0f} s", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, flush=True)
S2 = U2[["unit", "file", "recording", "depth", "animal"]].merge(pd.DataFrame(rows2), on=["unit", "recording"])
S2["layer"] = S2.depth.apply(L.layer_of)
print(len(S2), "unit-sets;", S2.unit.nunique(), "units")

In [ ]:
t0 = time.time()
rng_sh = np.random.default_rng(20190802)
for i, r in S2.iterrows():
    d = CUR[(r.unit, r.set)]
    csi = L.sparseness(d["curves"].T)
    sh = L.shuffle_csi(d["y"], d["comb"], d["dti"], d["kidx"], KER, rng_sh)
    p, dm = L.mahalanobis_p(csi, sh)
    d["csi_trace"] = csi
    S2.loc[i, "p_shuffle"] = p
    S2.loc[i, "maha"] = dm
print(f"shuffle tests: {time.time() - t0:.0f} s")

### exp2-units

> In experiment 2, 247 single units were isolated in 12 mice; 224 were responsive to correlated and 227 to anti-correlated stimuli.

In [ ]:
print(f"units in the files: {len(U2)} in {U2.file.nunique()} files; after dropping the duplicate: {S2.unit.nunique()}")
rows = []
for o2 in "ab":
    for sr in "ab":
        resp = S2[S2[f"p_{o2}{sr}"] < 0.05]
        rows.append(dict(directions=o2, sham=sr, corr=int((resp.set == "corr").sum()), anti=int((resp.set == "anti").sum()),
                         sets=len(resp)))
print(pd.DataFrame(rows).to_string(index=False))
RESP = (S2.p_aa < 0.05)  # main reading: O2 (a), sham (a)
print("layers of responsive sets (O2 a, sham a):", S2[RESP].groupby(["set", "layer"]).size().unstack().to_dict("index"))
print("expected layer counts corr/anti: L2/3 54/54, L4 78/79, L5 71/70, L6 21/24")
print("responsive sets per mouse:", S2[RESP].groupby("animal").size().to_dict())
print("recordings without sham trials (tested on the nine whiskers alone):",
      [r for r, t in T2.items() if (t.kind == "sham").sum() == 0])

**Result: reproduced.** The files hold 247 units in 12 mice (246 after dropping a duplicated unit); under direction reading (a) and sham reading (a), 231 are responsive to correlated and 232 to anti-correlated stimuli, within tolerance of 224 and 227; the other three readings give 236 to 240, also within tolerance.

### methods-dt-modulation

> Of 3,608 dt tuning curves, 33% had monotonically increasing likelihoods and 23% were best fit unsmoothed, so 56% (2,004) were set constant across dt.

Reading (a) of the criterion (held-out dt values without training trials scored on the training mean, kernel renormalised at the edges) is the main run; reading (b), zero padding at the edges, is run below on the responsive sets.

The verifier asked how the "unsmoothed" category could arise. Two other ways of scoring a held-out trial at a dt with no training trials are run on the same folds (an assumption, as the Methods do not say): (c) such trials left out of every likelihood, (d) left out of the unsmoothed likelihood only, as a sum that skips the undefined points of the unsmoothed curve would do. The cell also gives where the cross-validated likelihood peaks, and how many curves have a likelihood that rises at every step of the kernel SD (the strict reading of "monotonically increased").

In [ ]:
def status_table(S, key="status"):
    st = np.concatenate([CUR[(u, s)][key] for u, s in zip(S.unit, S.set)])
    n = len(st)
    return dict(curves=n, monotonic=np.mean(st == "monotonic"), unsmoothed=np.mean(st == "unsmoothed"),
                constant=int(np.sum(st != "modulated")), constant_frac=np.mean(st != "modulated"))
print("responsive sets (O2 a, sham a):", {k: round(v, 3) for k, v in status_table(S2[RESP]).items()})
print("all sets:", {k: round(v, 3) for k, v in status_table(S2).items()})
LL = [ll for u, s in zip(S2[RESP].unit, S2[RESP].set) for ll in CUR[(u, s)]["ll"]]
for lab, ks, kr in [("(c) left out of every likelihood", "sm_drop", "raw_drop"), ("(d) left out of the unsmoothed likelihood only", "sm", "raw_drop")]:
    st_ = np.array([L.dt_status(ll[ks], ll[kr]) for ll in LL])
    print(f"reading {lab}: monotonic {np.mean(st_ == 'monotonic'):.3f}, unsmoothed {np.mean(st_ == 'unsmoothed'):.3f}, "
          f"constant {np.sum(st_ != 'modulated')} ({np.mean(st_ != 'modulated'):.3f})")
best_k = np.array([np.argmax(ll["sm"]) for ll in LL])
strict = np.array([np.all(np.diff(ll["sm"]) > 0) for ll in LL])
gap = np.array([ll["raw"] - ll["sm"].max() for ll in LL])
print(f"reading (a): likelihood peaks at the smallest SD (0.5 ms) for {np.mean(best_k == 0):.3f}, at the largest (30 ms) for {np.mean(best_k == len(L.SDS) - 1):.3f}; "
      f"rises at every step for {np.mean(strict):.3f}; unsmoothed minus best smoothed log likelihood: median {np.median(gap):.1f}, largest {gap.max():.1f}")

In [ ]:
t0 = time.time()
KERZ = L.gaussian_kernels_zeropad(101, L.SDS)
rng_z = np.random.default_rng(20190803)
stz = []
for _, r in S2[RESP].iterrows():
    d = CUR[(r.unit, r.set)]
    for ci in range(8):
        sel_ = d["comb"] == ci
        stz.append(L.fast_dt_smooth(d["y"][sel_], d["dti"][sel_], rng_z, KERZ)[2])
stz = np.array(stz)
print(f"reading (b), zero padding: curves {len(stz)}, monotonic {np.mean(stz == 'monotonic'):.3f}, unsmoothed {np.mean(stz == 'unsmoothed'):.3f}, "
      f"constant {np.sum(stz != 'modulated')} ({np.mean(stz != 'modulated'):.3f}); {time.time() - t0:.0f} s")
print("expected: monotonic 0.33 +- 0.033, unsmoothed 0.23 +- 0.023, constant 2004 +- 200.4")

**Result: not reproduced.** Of the 3,704 CW-SW curves of the responsive sets, under reading (a) 0.574 have a likelihood that is largest at the largest kernel, none is best fit unsmoothed, and 2,127 (0.574) are set constant: the total count is within tolerance of 2,004 but the two fractions are far from 0.33 and 0.23. Under reading (b) (zero padding) only 39 curves (0.011) are set constant. The two other ways of scoring held-out trials at a dt without training trials, (c) and (d), give the same 0.574 and no unsmoothed curve. The unsmoothed likelihood is below the best smoothed one for every curve (by 102 at the least, median 10,667), the likelihood never peaks at the smallest SD, and 0.471 of curves rise at every step of the SD.

### exp2-combination-selective-prevalence

> 52% of experiment 2 units (129) were significantly combination-selective, based on correlated (35%) or anti-correlated stimuli (46%); 187 response sets were combination-selective and 264 nonselective.

In [ ]:
R2 = S2[RESP].copy().reset_index(drop=True)
R2["sel_fdr_all"] = L.bh(R2.p_shuffle.to_numpy())
R2["sel_fdr_sets"] = False
for s_ in ["corr", "anti"]:
    m = (R2.set == s_).to_numpy()
    R2.loc[m, "sel_fdr_sets"] = L.bh(R2.loc[m, "p_shuffle"].to_numpy())
ncell = S2.unit.nunique()
rows = []
for fam in ["sel_fdr_all", "sel_fdr_sets"]:
    sel = R2[R2[fam]]
    cells = sel.unit.nunique()
    nc, na = int((sel.set == "corr").sum()), int((sel.set == "anti").sum())
    rc, ra = int((R2.set == "corr").sum()), int((R2.set == "anti").sum())
    rows.append(dict(fdr_family=fam, selective_sets=len(sel), nonselective_sets=len(R2) - len(sel), cells=cells,
                     frac_cells=cells / ncell, corr_over_resp=nc / rc, anti_over_resp=na / ra,
                     corr_over_cells=nc / ncell, anti_over_cells=na / ncell))
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("uncorrected p < 0.05:", int((R2.p_shuffle < 0.05).sum()), "of", len(R2))
print("selective sets per mouse (FDR all):", R2[R2.sel_fdr_all].groupby("animal").size().to_dict())
SEL = R2.sel_fdr_all.to_numpy()  # main reading (O4 a)

Calibration of the test on the real data (the verifier's request). For each responsive set, the counts are permuted across the eight combinations within each dt, which removes combination tuning and keeps everything else, and the whole test is run on the result: kernels chosen again by cross-validation on the permuted counts, then the shuffles (1000 here, to keep the run short) and the Mahalanobis p-value. A calibrated test gives p < 0.05 for about 5% of these sets. The measured p-values are then calibrated on this null distribution (the share of null p-values at or below each), and a group of the paper's size, the 187 sets with the smallest p, is defined for the variant section below.

In [ ]:
t0 = time.time()
rng_null = np.random.default_rng(20190804)
p_null = []
for _, r in R2.iterrows():
    d = CUR[(r.unit, r.set)]
    y0 = L.permute_within_dt(d["y"], d["comb"], d["dti"], rng_null)
    p_null.append(L.shuffle_test(y0, d["comb"], d["dti"], rng_null, KER, n_shuf=1000)[0])
p_null = np.array(p_null)
print(f"real-data null, {len(p_null)} sets ({time.time() - t0:.0f} s): p < 0.05 for {np.mean(p_null < 0.05):.3f}, p < 0.01 for {np.mean(p_null < 0.01):.3f}, "
      f"p at its floor (1/1001) for {np.mean(p_null <= 1 / 1001):.3f}; selective after FDR: {int(L.bh(p_null).sum())}")
edges = [0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0001]
print("histogram of p by bin", [f"{a}-{b:.3g}" for a, b in zip(edges[:-1], edges[1:])])
print("   measured:", np.histogram(R2.p_shuffle, edges)[0].tolist())
print("   null:    ", np.histogram(p_null, edges)[0].tolist())
p_cal = np.array([(1 + np.sum(p_null <= p)) / (1 + len(p_null)) for p in R2.p_shuffle])
print(f"measured p calibrated on the null p: calibrated p < 0.05 for {int(np.sum(p_cal < 0.05))} sets; selective after FDR: {int(L.bh(p_cal).sum())} "
      f"(the null's floor of 1/1001 limits how small a calibrated p can be)")
CAL = L.bh(p_cal)  # the calibrated group: FDR on the calibrated p-values
print(f"calibrated group: {int(CAL.sum())} sets, correlated {int((R2.set[CAL] == 'corr').sum())}, anti-correlated {int((R2.set[CAL] == 'anti').sum())}, "
      f"cells {R2.unit[CAL].nunique()}; in the uncalibrated selective group {int((CAL & SEL).sum())}")
order = np.lexsort((-R2.maha.to_numpy(), R2.p_shuffle.to_numpy()))
TOP187 = np.zeros(len(R2), bool)
TOP187[order[:187]] = True
print(f"variant group of the paper's size: the 187 sets with the smallest p (ties broken by the larger Mahalanobis distance); "
      f"largest p in it {R2.p_shuffle[TOP187].max():.4f}; sets at the smallest p ({R2.p_shuffle.min():.5f}): {int((R2.p_shuffle == R2.p_shuffle.min()).sum())}; "
      f"correlated {int((R2.set[TOP187] == 'corr').sum())}, anti-correlated {int((R2.set[TOP187] == 'anti').sum())}, cells {R2.unit[TOP187].nunique()}")

**Result: not reproduced.** The shuffle test finds 318 of 463 responsive sets combination-selective (FDR over all sets; 312 with the two sets apart), in 189 of 246 cells (0.768), against 187 sets and 129 cells (0.52). The fractions are 0.641 and 0.733 of the responsive correlated and anti-correlated sets, or 0.602 and 0.691 of all cells, against 0.35 and 0.46. The test is not calibrated here: on the real-data null it gives p < 0.05 for 0.246 of sets (0.076 below 0.01), although FDR control then selects none of them. Calibrated on that null, the measured p-values give 162 selective sets after FDR. The 187 sets with the smallest p, used in the variant section below, lie in 130 cells and are 74 correlated and 113 anti-correlated sets; their count is fixed at the paper's, so they do not test this claim.

### Per-unit measures used by the claims below

For each responsive set (a "unit" in the paper's sense): the best combination and best dt (the maximum of the cwSTRF), the eight responses at the best dt under O3 (a) the smoothed value and (b) the mean raw count within +-5 ms of the best dt, the linear predictions under O2 (a) direction-specific and (b) both directions, CW or SW tuning under the same two readings, and the spontaneous rate. A set whose best-combination curve was set constant across dt has no defined best dt (the argmax falls on the first tied dt, -50 ms); it is flagged, and its eight responses are taken over all dt: under O3 (a) the mean of each smoothed curve over dt, under O3 (b) the mean count over all of the combination's trials, and its decoders use all its trials (an assumption: the best curve is flat, so every dt is equally the best, and the mean over dt does not favour one).

In [ ]:
def derive(R):
    out = []
    for _, r in R.iterrows():
        d = CUR[(r.unit, r.set)]
        cv = d["curves"]
        c_best, j_best = np.unravel_index(np.argmax(cv), cv.shape)
        const = d["status"][c_best] != "modulated"
        # A constant best curve has no best dt (assumption): the responses are taken over all dt.
        resp_a = cv.mean(axis=1) if const else cv[:, j_best]
        near = np.ones(len(d["y"]), bool) if const else np.abs(d["dti"] - j_best) <= 5
        resp_b = np.array([d["y"][near & (d["comb"] == c)].mean() if np.any(near & (d["comb"] == c)) else np.nan for c in range(8)])
        sm = d["single_means"]
        dirs = d["dirs"]
        sing_a = np.array([sm.loc[[(w, x) for x in sorted(dirs[w])]].mean() for w in range(1, 10)])
        sing_b = np.array([sm.loc[w].mean() for w in range(1, 10)])
        out.append(dict(c_best=c_best, j_best=j_best, dt_best=j_best - 50, best_const=const,
                        resp_a=resp_a, resp_b=resp_b, lin_a=d["lin_a"], lin_b=d["lin_b"],
                        cw_tuned_a=np.argmax(sing_a) + 1 == L.CW12, cw_tuned_b=np.argmax(sing_b) + 1 == L.CW12,
                        sing_a=sing_a, sing_b=sing_b,
                        csi_a=L.sparseness(resp_a), csi_b=L.sparseness(resp_b),
                        csi_lin_a=L.sparseness(d["lin_a"]), csi_lin_b=L.sparseness(d["lin_b"])))
    df = pd.DataFrame(out)
    for c in ["csi_a", "csi_b", "csi_lin_a", "csi_lin_b"]:
        df[c] = df[c].astype(float)
    for c in ["c_best", "j_best", "dt_best"]:
        df[c] = df[c].astype(int)
    for c in ["best_const", "cw_tuned_a", "cw_tuned_b"]:
        df[c] = df[c].astype(bool)
    return pd.concat([R.reset_index(drop=True), df], axis=1)

M2 = derive(R2)
M2["sel"] = SEL
M2v = M2.assign(sel=TOP187)  # the variant group of the paper's size
M2c = M2.assign(sel=CAL)  # the calibrated group, from the real-data null
print(len(M2), "sets;", int(M2.sel.sum()), "selective;", int(M2.best_const.sum()), "with a constant best-combination curve")

### fig2f-shuffle-abolishes-tuning

> Average peak-aligned combination tuning of combination-selective units was sharp, and shuffling spike counts across stimuli and trials abolished it.

In [ ]:
def run_fig2f_shuffle_abolishes_tuning(M2):
    rng_2f = np.random.default_rng(2)
    selM = M2[M2.sel]
    meas, sh_a, sh_b = [], [], []
    for _, r in selM.iterrows():
        d = CUR[(r.unit, r.set)]
        order = np.r_[r.c_best, [c for c in range(8) if c != r.c_best]]
        meas.append(r.resp_a[order])
        y_sh = d["y"].copy()
        for j in np.unique(d["dti"]):  # one shuffle across combinations within each dt
            ii = np.where(d["dti"] == j)[0]
            y_sh[ii] = y_sh[rng_2f.permutation(ii)]
        curves = np.zeros((8, 101))
        for c in range(8):
            sel_ = d["comb"] == c
            raw = np.bincount(d["dti"][sel_], weights=y_sh[sel_], minlength=101) / np.maximum(np.bincount(d["dti"][sel_], minlength=101), 1)
            curves[c] = KER[d["kidx"][c]] @ raw if d["kidx"][c] is not None else np.full(101, y_sh[sel_].mean())
        v = curves.mean(axis=1) if r.best_const else curves[:, r.j_best]
        sh_a.append(v[order])
        sh_b.append(np.r_[v.max(), np.delete(v, v.argmax())])
    meas, sh_a, sh_b = map(np.array, (meas, sh_a, sh_b))
    for name, arr in [("measured", meas), ("shuffled, aligned to measured best (a)", sh_a), ("shuffled, own peak (b)", sh_b)]:
        m = arr.mean(0)
        print(f"{name}: mean curve {m.round(3)}, CSI of mean curve {L.sparseness(m):.3f}")
    first, rest = sh_a[:, 0], sh_a[:, 1:].mean(1)
    diff = first - rest
    print(f"(a) shuffled first point minus mean of the other seven: {diff.mean():.4f} (s.e.m. {diff.std(ddof=1) / np.sqrt(len(diff)):.4f}); "
          f"s.e.m. of the first point {first.std(ddof=1) / np.sqrt(len(first)):.4f}; n = {len(diff)}")

run_fig2f_shuffle_abolishes_tuning(M2)

**Result: reproduced.** Over the 318 selective sets the mean peak-aligned tuning has a CSI of 0.033 (peak 0.861 against 0.516 to 0.607 for the other combinations); one shuffle per set gives a flat curve (CSI 0.000 aligned to the measured best combination, 0.003 aligned to its own peak), and the shuffled first point exceeds the mean of the other seven by 0.011, within the s.e.m. of that point (0.027).

### fig2f-most-selective-lower-firing

> Among combination-selective units, the 25% most selective had lower spontaneous and whisker-evoked firing than the 25% least selective.

In [ ]:
def run_fig2f_most_selective_lower_firing(M2):
    selM = M2[M2.sel]
    q = selM.sort_values("csi_a")
    nq = len(q) // 4
    lo, hi = q.iloc[:nq], q.iloc[-nq:]
    rows = []
    for lab, f in [("spont, sham a (Hz)", lambda d: d.spont_a), ("spont, sham b (Hz)", lambda d: d.spont_b),
                   ("evoked (a): best response", lambda d: d.resp_a.apply(np.max)), ("evoked (b): mean of 8", lambda d: d.resp_a.apply(np.mean))]:
        a, b = f(hi).dropna(), f(lo).dropna()
        t = stats.ttest_ind(a, b)
        rows.append(dict(measure=lab, most_selective=a.mean(), least_selective=b.mean(), t=t.statistic, p=t.pvalue, n=f"{len(a)}/{len(b)}"))
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

run_fig2f_most_selective_lower_firing(M2)

**Result: reproduced.** The most selective quarter (79 sets) has lower spontaneous firing (0.84 against 3.87 Hz, p = 1.4e-12) and lower evoked firing (best response 0.63 against 1.22 spikes per stimulus, p = 6.7e-9; mean of eight 0.32 against 0.99) than the least selective quarter.

### supp4-selectivity-at-high-rates

> Substantial combination selectivity was present among the most responsive combination-selective units and after subtracting spontaneous activity.

In [ ]:
def run_supp4_selectivity_at_high_rates(M2):
    for o3 in "ab":
        bestr = M2[f"resp_{o3}"].apply(np.nanmax)
        print(f"O3 ({o3}): selective < 0.81:", int(((bestr < 0.81) & M2.sel).sum()), "| selective >= 0.81:", int(((bestr >= 0.81) & M2.sel).sum()),
              "| nonselective >= 0.81:", int(((bestr >= 0.81) & ~M2.sel).sum()), "| nonselective < 0.81:", int(((bestr < 0.81) & ~M2.sel).sum()))
        g = M2[(bestr >= 0.81) & M2.sel]
        al = []
        for _, r in g.iterrows():
            order = np.r_[r.c_best, [c for c in range(8) if c != r.c_best]]
            sp = r.spont_a * L.WINDOW / 1000 if np.isfinite(r.spont_a) else np.nan
            al.append(r[f"resp_{o3}"][order] - sp)
        al = np.array(al)
        al = al[np.isfinite(al).all(1)]
        pk = al[:, 0] - al[:, 1:].mean(1)
        print(f"   high-rate selective, spontaneous subtracted (n = {len(al)}, sets with sham trials): mean curve {al.mean(0).round(3)}")
        print(f"   peak minus mean of other seven: {pk.mean():.3f}, s.e.m. {pk.std(ddof=1) / np.sqrt(len(pk)):.3f}, ratio {pk.mean() / (pk.std(ddof=1) / np.sqrt(len(pk))):.1f}")

run_supp4_selectivity_at_high_rates(M2)

**Result: partially reproduced.** Among the 128 high-rate selective sets with sham trials, the spontaneous-subtracted peak-aligned tuning has a peak 0.452 above the mean of the other seven combinations, 22.9 times its s.e.m. (O3 b: 0.478, 20.0 times). The group counts are not reproduced: 178 selective sets below 0.81 spikes per stimulus, 140 at or above, and 28 nonselective at or above (O3 b: 175, 143 and 38), against 95, 92 and 91.

### fig2g-best-combination-bias

> Best CW-SW combinations covered all combinations but were nonuniformly distributed, biased toward the rostral within-row SW and the SWs ventral to the CW (Rayleigh test, P = 4 x 10-19, n = 451).

In [ ]:
def rayleigh(theta):
    n = len(theta)
    Rbar = np.abs(np.mean(np.exp(1j * theta)))
    Z = n * Rbar ** 2
    p = np.exp(np.sqrt(1 + 4 * n + 4 * (n ** 2 - (n * Rbar) ** 2)) - (1 + 2 * n))  # Zar's approximation
    return Z, min(p, 1.0)
bw = np.array(L.SWS12)[M2.c_best]
rel = [L.relpos(w, 2) for w in bw]
theta = np.array([np.arctan2(-dr, dc) for dr, dc in rel])  # angle on the face: rostral = 0, dorsal = 90 degrees
Z, p = rayleigh(theta)
cnt = pd.Series([L.POSNAME[x] for x in rel]).value_counts()
print(cnt.to_string())
print(f"Rayleigh Z = {Z:.2f}, p = {p:.2g}, n = {len(theta)}")
dors = sum(cnt.get(x, 0) for x in ["dorsal same-arc", "dorsal rostral", "dorsal caudal"])
vent = sum(cnt.get(x, 0) for x in ["ventral same-arc", "ventral rostral", "ventral caudal"])
print(f"every SW best at least once: {len(cnt) == 8}; most frequent: {cnt.index[0]}; ventral row {vent} vs dorsal row {dors}")
print("by set:", M2.groupby("set").apply(lambda d: pd.Series([L.POSNAME[L.relpos(w, 2)] for w in np.array(L.SWS12)[d.c_best]]).value_counts().idxmax()).to_dict())

**Result: reproduced.** Best combinations of the 463 responsive sets are nonuniform (Rayleigh p = 1.1e-22); every SW is best for at least 14 sets, the rostral within-row SW is the most frequent (150), and the ventral row (210) is more frequent than the dorsal row (63).

### fig3a-sharper-than-linear

> CW-SW combination tuning was sharper than predicted from the linear sum of single-whisker responses for nearly all units (P = 1 x 10-50, paired t-test).

In [ ]:
rows = []
for o3 in "ab":
    for o2 in "ab":
        m, l = M2[f"csi_{o3}"], M2[f"csi_lin_{o2}"]
        t = stats.ttest_rel(m, l)
        rows.append(dict(O3=o3, O2=o2, n=len(m), measured=m.mean(), linear=l.mean(), frac_above=np.mean(m > l), t=t.statistic, p=t.pvalue))
fmt = lambda v: f"{v:.4g}"
print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

**Result: reproduced.** Measured CSI at the best dt (0.084) is above the linear CSI (0.035) by paired t-test (p = 1.8e-41), with 0.903 of 463 sets above under readings O3 (a) and O2 (a); the other readings give 0.942 to 0.972 above and p below 1e-44.

### fig3b-sharper-in-both-groups

> Measured CSI exceeded linearly predicted CSI in both combination-selective (187) and nonselective (264) units.

In [ ]:
def run_fig3b_sharper_in_both_groups(M2):
    rows = []
    for grp, g in [("selective", M2[M2.sel]), ("nonselective", M2[~M2.sel])]:
        for o3, o2 in [("a", "a"), ("a", "b"), ("b", "a"), ("b", "b")]:
            m, l = g[f"csi_{o3}"], g[f"csi_lin_{o2}"]
            rows.append(dict(group=grp, O3=o3, O2=o2, n=len(g), measured=m.mean(), linear=l.mean(),
                             p_two_sample=stats.ttest_ind(m, l).pvalue, p_paired=stats.ttest_rel(m, l).pvalue))
    print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

run_fig3b_sharper_in_both_groups(M2)

**Result: reproduced.** Measured CSI exceeds linear CSI in the 318 selective sets (0.091 against 0.040, two-sample p = 4.8e-18) and in the 145 nonselective sets (0.069 against 0.023, p = 7.6e-8), under every reading.

### fig3b-linear-sharper-in-selective

> The linearly predicted receptive field was sharper in combination-selective than in nonselective units.

In [ ]:
def run_fig3b_linear_sharper_in_selective(M2):
    for o2 in "ab":
        a, b = M2.loc[M2.sel, f"csi_lin_{o2}"], M2.loc[~M2.sel, f"csi_lin_{o2}"]
        t = stats.ttest_ind(a, b)
        print(f"O2 ({o2}): linear CSI selective {a.mean():.4f} (n={len(a)}), nonselective {b.mean():.4f} (n={len(b)}), t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

run_fig3b_linear_sharper_in_selective(M2)

**Result: reproduced.** The linear CSI is higher in selective sets (0.0399 against 0.0234, p = 7.3e-5; 0.0312 against 0.0168, p = 3.7e-6 under O2 b).

### supp6-layers

> Combination selectivity existed in all layers but was stronger in L2-4 than in L5-6 (one-factor ANOVA, P = 0.0025).

In [ ]:
def run_supp6_layers(M2):
    print("units (sets) by layer, all responsive:", M2.layer.value_counts().to_dict(), "; expected 108 / 157 / 141 / 45")
    for lab, g in [("(a) all responsive sets", M2), ("(b) selective sets", M2[M2.sel])]:
        g = g[g.layer != "outside"]
        vals = {lay: np.log10(g.loc[g.layer == lay, "csi_a"].clip(lower=1e-6)) for lay in ["L2/3", "L4", "L5", "L6"]}
        f = stats.f_oneway(*vals.values())
        print(lab, {k: (round(v.mean(), 3), len(v)) for k, v in vals.items()}, f"F = {f.statistic:.2f}, p = {f.pvalue:.3g}")
    print("sets outside the layer bounds:", int((M2.layer == "outside").sum()), "; CSI = 0 sets (floored at 1e-6 for the log):", int((M2.csi_a <= 0).sum()))

run_supp6_layers(M2)

**Result: partially reproduced.** Log10 CSI differs across layers (ANOVA p = 3.2e-5 over all responsive sets, p = 0.0015 over selective sets), with L2/3 (-1.15) and L4 (-1.25) above L5 (-1.41) and L6 (-1.35). The layer counts are 103, 159 and 147 for L2/3 to L5, within tolerance, and 53 for L6, outside 45 +- 4.5.

### fig4a-sw-tuned-sharper

> SW-tuned units were more sharply combination-tuned than CW-tuned units (P = 3.4 x 10-4, two-sample t-test; 169 CW-tuned, 282 SW-tuned).

In [ ]:
for o2 in "ab":
    cw = M2[f"cw_tuned_{o2}"].to_numpy(bool)
    a, b = M2.loc[~cw, "csi_a"], M2.loc[cw, "csi_a"]
    t = stats.ttest_ind(a, b)
    print(f"O2 ({o2}): CW-tuned {cw.sum()}, SW-tuned {(~cw).sum()}; CSI SW-tuned {a.mean():.4f}, CW-tuned {b.mean():.4f}, t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** Under O2 (a), 174 sets are CW-tuned and 289 SW-tuned (within tolerance of 169 and 282), and SW-tuned sets are more sharply combination-tuned (CSI 0.101 against 0.056, p = 3.4e-7); O2 (b) gives 216 and 247 with p = 6.1e-5.

### fig5a-global-scaling

> Across all stimuli and units, CW-SW responses at the best dt were well predicted by a 64% scaling of the linear sum of single-whisker responses (R2 = 0.90, n = 3,608 combinations).

In [ ]:
rows = []
for o3 in "ab":
    for o2 in "ab":
        y = np.concatenate(M2[f"resp_{o3}"].to_numpy())
        x = np.concatenate(M2[f"lin_{o2}"].to_numpy())
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        b = (x @ y) / (x @ x)
        sse = np.sum((y - b * x) ** 2)
        rows.append(dict(O3=o3, O2=o2, n=len(x), slope=b, R2_centred=1 - sse / np.sum((y - y.mean()) ** 2), R2_uncentred=1 - sse / np.sum(y ** 2)))
F5 = pd.DataFrame(rows)
print(F5.to_string(index=False, float_format=fmt))
print("ranges: measured", np.concatenate(M2.resp_a.to_numpy()).min().round(3), "-", np.concatenate(M2.resp_a.to_numpy()).max().round(3),
      "; linear", np.concatenate(M2.lin_a.to_numpy()).min().round(3), "-", np.concatenate(M2.lin_a.to_numpy()).max().round(3))
SLOPE = 0.64  # the paper's scaling, used by the Fig. 6 claims as the criteria say

**Result: reproduced.** The least-squares slope through the origin of the 3,704 measured responses on their linear predictions is 0.636, with R2 = 0.926 (centred) or 0.967 (uncentred); the other readings give slopes 0.620 to 0.643 and R2 0.876 to 0.967.

### fig5bc-linearity-index

> Nonoptimal combinations had a linearity index of 0.63 +- 0.003 and best combinations 0.95 +- 0.02 (P = 1.3 x 10-147, two-sample t-test).

In [ ]:
rows = []
for o3 in "ab":
    for o2 in "ab":
        li = [r[f"resp_{o3}"] / r[f"lin_{o2}"] for _, r in M2.iterrows()]
        best = np.array([v[c] for v, c in zip(li, M2.c_best)])
        sub = np.concatenate([np.delete(v, c) for v, c in zip(li, M2.c_best)])
        best, sub = best[np.isfinite(best)], sub[np.isfinite(sub)]
        t = stats.ttest_ind(best, sub)
        rows.append(dict(O3=o3, O2=o2, best=best.mean(), best_sem=stats.sem(best), nonopt=sub.mean(), nonopt_sem=stats.sem(sub),
                         n_best=len(best), n_nonopt=len(sub), t=t.statistic, p=t.pvalue))
print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

**Result: partially reproduced.** The linearity index is 0.909 +- 0.021 for best combinations and 0.647 +- 0.003 for nonoptimal ones (p = 1.2e-121), against 0.95 +- 0.02 and 0.63 +- 0.003; both means miss their tolerance and lie within 10% of the reported values, and the ordering and test hold. Reading O3 (b) with O2 (a) puts the best mean (0.964) within tolerance but not the nonoptimal one (0.655).

### fig5c-pseudo-unit-shuffle

> Pseudo-units made by sampling eight random points from Fig. 5a had very similar linearity for their strongest and weaker responses (0.71 against 0.67), so the best-combination linearity is not a trivial consequence of response strength.

In [ ]:
rng_5c = np.random.default_rng(5)
y_all = np.concatenate(M2.resp_a.to_numpy()); x_all = np.concatenate(M2.lin_a.to_numpy())
for npu in [len(M2), 10000]:
    bl, sl = [], []
    for _ in range(npu):
        ii = rng_5c.choice(len(y_all), 8, replace=False)
        li = y_all[ii] / x_all[ii]
        c = np.argmax(y_all[ii])
        bl.append(li[c]); sl.extend(np.delete(li, c))
    print(f"{npu} pseudo-units: strongest {np.mean(bl):.3f}, remaining {np.mean(sl):.3f}")

**Result: reproduced.** Over 463 pseudo-units the strongest responses have a linearity of 0.680 and the remaining ones 0.673, within tolerance of 0.71 and 0.67 (0.683 and 0.680 over 10,000 pseudo-units).

### fig5c-sublinear-at-all-dt

> Sublinearity for nonoptimal combinations held at all dt values, not only at the best dt.

In [ ]:
peak_sub, best_li = [], []
for _, r in M2.iterrows():
    cv = CUR[(r.unit, r.set)]["curves"]
    li = cv / r.lin_a[:, None]
    peak_sub.extend(np.delete(li.max(axis=1), r.c_best))
    best_li.append(r.resp_a[r.c_best] / r.lin_a[r.c_best])
peak_sub, best_li = np.array(peak_sub), np.array(best_li)
t = stats.ttest_ind(peak_sub, best_li)
print(f"peak linearity index over dt, nonoptimal: mean {peak_sub.mean():.3f} (n={len(peak_sub)}); best combination at best dt {best_li.mean():.3f} (n={len(best_li)}); t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** The peak linearity index over dt of the 3,241 nonoptimal combinations has a mean of 0.703, below 1 and below the best-combination mean of 0.909 (p = 2e-61).

### supralinear-best-8-percent

> Only 8% of units had significant supralinearity for the best combination.

In [ ]:
def run_supralinear_best_8_percent(M2):
    rows = []
    for o3 in "ab":
        zs, dfs = [], []
        for _, r in M2.iterrows():
            d = CUR[(r.unit, r.set)]
            sm_, sv, sn = d["single_means"], d["single_vars"], d["single_n"]
            dirs = d["dirs"]
            w = L.SWS12[r.c_best]
            def m_se(wh):
                keys = [(wh, x) for x in sorted(dirs[wh])]
                mu = sm_.loc[keys].mean()
                se2 = np.mean([sv.loc[k_] / sn.loc[k_] for k_ in keys]) / len(keys)
                return mu, se2
            (mc, sc), (ms, ss) = m_se(L.CW12), m_se(w)
            mu = r[f"resp_{o3}"][r.c_best]
            zs.append((mu - (mc + ms)) / np.sqrt(sc + ss))
            near = ((np.abs(d["dti"] - r.j_best) <= 5) | r.best_const) & (d["comb"] == r.c_best)
            dfs.append(max(near.sum() - 1, 1))
        zs, dfs = np.array(zs), np.array(dfs)
        for ref, pv in [("normal", stats.norm.sf(zs)), ("t", stats.t.sf(zs, dfs))]:
            for grp, m in [("all", np.ones(len(M2), bool)), ("selective", M2.sel.to_numpy())]:
                rows.append(dict(O3=o3, reference=ref, units=grp, n=int(m.sum()), frac_supralinear=np.mean(pv[m] < 0.05)))
    print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

run_supralinear_best_8_percent(M2)

**Result: reproduced.** Under O3 (a), 0.072 of the 318 selective sets (reading d) are significantly supralinear for their best combination, inside the tolerance band (0.072 to 0.088) at its lower edge, by either reference distribution; over all 463 sets the fraction is 0.093, and O3 (b) gives 0.129 and 0.151. The outcome sits on the band's edge. Over the 187 sets with the smallest shuffle p (variant section), O3 (b) gives 0.080, within tolerance, and O3 (a) 0.032. On the 162 calibrated sets (calibrated-group section) no reading is within tolerance (0.031 and 0.093 of selective sets), so the outcome also rests on the selective group. The fraction is a count, 23 of 318 sets: one set fewer would put it below the band.

### fig6b-best-above-worst-below-scaled

> In both combination-selective and nonselective units, the best combination exceeded the 0.64-scaled linear prediction and the worst combination fell below it.

In [ ]:
SLOPE = 0.64  # the paper's global scaling (Results, Fig. 6), as the criteria say
def ranked_resid(g, o3="a", o2="a"):
    res, lin_, meas_ = [], [], []
    for _, r in g.iterrows():
        o = np.argsort(-r[f"resp_{o3}"])
        res.append(r[f"resp_{o3}"][o] - SLOPE * r[f"lin_{o2}"][o])
        meas_.append(r[f"resp_{o3}"][o]); lin_.append(SLOPE * r[f"lin_{o2}"][o])
    return np.array(res), np.array(meas_), np.array(lin_)
def run_fig6b_best_above_worst_below_scaled(M2):
    RES = {}
    for grp, g in [("selective", M2[M2.sel]), ("nonselective", M2[~M2.sel])]:
        res, me, li = ranked_resid(g)
        RES[grp] = res
        tb, tw = stats.ttest_rel(me[:, 0], li[:, 0]), stats.ttest_rel(me[:, -1], li[:, -1])
        print(f"{grp} (n={len(g)}): best {me[:, 0].mean():.3f} vs 0.64 x linear {li[:, 0].mean():.3f}, t = {tb.statistic:.2f}, p = {tb.pvalue:.3g}; "
              f"worst {me[:, -1].mean():.3f} vs {li[:, -1].mean():.3f}, t = {tw.statistic:.2f}, p = {tw.pvalue:.3g}")
        print("   mean residual by rank:", res.mean(0).round(3))

run_fig6b_best_above_worst_below_scaled(M2)

**Result: reproduced.** The best combination exceeds the 0.64-scaled linear prediction in selective (p = 1e-37) and nonselective sets (p = 3e-19), and the worst combination falls below it (p = 7e-26 and 3e-7).

### fig6b-enhancement-not-different

> The magnitude of combination-specific enhancement for the best combination did not differ between selective and nonselective units (P = 0.24, t-test).

In [ ]:
def run_fig6b_enhancement_not_different(M2):
    RES = {grp: ranked_resid(g)[0] for grp, g in [("selective", M2[M2.sel]), ("nonselective", M2[~M2.sel])]}
    a, b = RES["selective"][:, 0], RES["nonselective"][:, 0]
    t = stats.ttest_ind(a, b)
    print(f"rank-1 residual: selective {a.mean():.4f} (n={len(a)}), nonselective {b.mean():.4f} (n={len(b)}), difference {a.mean() - b.mean():.4f}, t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
    for o3, o2 in [("b", "a"), ("a", "b")]:
        ra = ranked_resid(M2[M2.sel], o3, o2)[0][:, 0]; rb = ranked_resid(M2[~M2.sel], o3, o2)[0][:, 0]
        print(f"variant O3 {o3}, O2 {o2}: difference {ra.mean() - rb.mean():.4f}, p = {stats.ttest_ind(ra, rb).pvalue:.3g}")

run_fig6b_enhancement_not_different(M2)

**Result: not reproduced.** The rank-1 enhancement is 0.157 in selective and 0.112 spikes per stimulus in nonselective sets, a difference of 0.045 that the two-sample t-test finds (p = 0.0096); the variants give p = 0.047 and 0.004.

### fig6b-suppression-greater-selective

> Suppression of the two weakest CW-SW combinations was modestly greater in selective units (P = 0.001 and P = 0.002, two-sample t-tests, Bonferroni).

In [ ]:
def run_fig6b_suppression_greater_selective(M2):
    RES = {grp: ranked_resid(g)[0] for grp, g in [("selective", M2[M2.sel]), ("nonselective", M2[~M2.sel])]}
    for rank in [7, 8]:
        a, b = RES["selective"][:, rank - 1], RES["nonselective"][:, rank - 1]
        t = stats.ttest_ind(a, b)
        print(f"rank {rank}: selective {a.mean():.4f}, nonselective {b.mean():.4f}, t = {t.statistic:.2f}, p = {t.pvalue:.3g}, "
              f"Bonferroni x8 {min(1, 8 * t.pvalue):.3g}, x2 {min(1, 2 * t.pvalue):.3g}")

run_fig6b_suppression_greater_selective(M2)

**Result: reproduced.** Residuals at ranks 7 and 8 are more negative in selective sets (-0.043 against -0.017 and -0.074 against -0.037), with Bonferroni-corrected p = 0.031 and 0.0033 over eight ranks (0.0077 and 0.0008 over two).

### fig6c-enhancement-strongest-sw

> Combination-selective units nonlinearly enhanced combinations involving the strongest single SW (P = 1.2 x 10-8, one-factor ANOVA), while nonselective units showed an opposite, much weaker trend (P = 0.03).

In [ ]:
def run_fig6c_enhancement_strongest_sw(M2):
    for grp, g in [("selective", M2[M2.sel]), ("nonselective", M2[~M2.sel])]:
        byrank = []
        for _, r in g.iterrows():
            sw_resp = r.sing_a[np.array(L.SWS12) - 1]
            o = np.argsort(-sw_resp)
            byrank.append((r.resp_a - SLOPE * r.lin_a)[o])
        byrank = np.array(byrank)
        f = stats.f_oneway(*byrank.T)
        m = byrank.mean(0)
        print(f"{grp} (n={len(g)}): mean residual by SW rank {m.round(4)}; ANOVA F = {f.statistic:.2f}, p = {f.pvalue:.3g}; "
              f"rank 1 largest {m.argmax() == 0}; rank 1 below mean of 2-8: {m[0] < m[1:].mean()}")

run_fig6c_enhancement_strongest_sw(M2)

**Result: partially reproduced.** In selective sets the residual differs with the rank of the SW's single-whisker response (ANOVA p = 4.7e-9) and is largest for the strongest SW (0.053). In nonselective sets the rank-1 residual (0.004) is below the mean of ranks 2 to 8, but the ANOVA does not find the trend (p = 0.18).

### fig7a-dt-half-width

> For their best combinations, combination-selective units had sharp dt tuning with an average half-width of about 10 ms around the best dt; suboptimal combinations had weaker, broader dt tuning.

In [ ]:
lags = np.arange(-100, 101)
def aligned(g):
    best, sub = [], []
    for _, r in g.iterrows():
        cv = CUR[(r.unit, r.set)]["curves"] / CUR[(r.unit, r.set)]["curves"].max()
        A = np.full((8, len(lags)), np.nan)
        A[:, 100 - r.j_best: 100 - r.j_best + 101] = cv
        best.append(A[r.c_best]); sub.extend(np.delete(A, r.c_best, axis=0))
    return np.array(best), np.array(sub)
def halfwidth(curve, mode):
    c = curve[np.isfinite(curve)] if False else curve
    pk = c[100]
    lvl = (pk + np.nanmin(c)) / 2 if mode == "halfway" else pk / 2
    left = np.where(np.isfinite(c[:100]) & (c[:100] <= lvl))[0]
    right = np.where(np.isfinite(c[101:]) & (c[101:] <= lvl))[0]
    if not len(left) or not len(right):
        return np.nan
    return ((100 - left.max()) + (right.min() + 1)) / 2
def run_fig7a_dt_half_width(M2):
    for lab, g in [("selective, all", M2[M2.sel]), ("selective, best curve dt-modulated", M2[M2.sel & ~M2.best_const])]:
        B, S = aligned(g)
        mb, ms = np.nanmean(B, 0), np.nanmean(S, 0)
        hw_a, hw_c = halfwidth(mb, "halfway"), halfwidth(mb, "half")
        hw_b = np.nanmean([halfwidth(b, "halfway") for b in B])
        print(f"{lab} (n={len(g)}): half-width (a) {hw_a}, (b) {hw_b:.1f} (units with a defined value: {np.sum(np.isfinite([halfwidth(b, 'halfway') for b in B]))}), (c) {hw_c} ms")
        print(f"   suboptimal mean at best dt {ms[100]:.3f} vs best {mb[100]:.3f}; suboptimal half-width (a) {halfwidth(ms, "halfway")}, (c) {halfwidth(ms, "half")} (nan: the curve does not fall to the level within +-100 ms)")
        print("   mean best curve at lags -30..30 step 5:", mb[70:131:5].round(3))

run_fig7a_dt_half_width(M2)

**Result: partially reproduced (resting on an assumption).** Over the 248 selective sets whose best-combination curve is dt-modulated, the mean aligned curve falls halfway from its peak to its minimum at a half-width of 11.5 ms (reading a), within 10 +- 2 ms, and the suboptimal mean is lower at the best dt (0.605 against 1.000); but the suboptimal mean does not fall to that level on both sides within +-100 ms, so its half-width is undefined and the criterion's comparison of half-widths cannot be made. Including the 70 sets whose best curve is constant, and so has no best dt, gives 16.5 ms (suboptimal 24.5 ms); the per-unit reading (b) gives 14.9 ms, averaged over only the 74 sets with a defined value; reading (c) is undefined because the mean curve stays above half its peak.

### fig7b-inbound-best-outbound-suboptimal

> Mean dt tuning had a minimum at -10 ms; best combinations evoked maximal responses at -30 to -50 ms (SW leads CW, inbound), suboptimal combinations at +30 to +50 ms (outbound).

In [ ]:
def run_fig7b_inbound_best_outbound_suboptimal(M2):
    for zmode in ["unit", "curve"]:
        zb, zs = [], []
        for _, r in M2[M2.sel].iterrows():
            cv = CUR[(r.unit, r.set)]["curves"]
            if zmode == "unit":
                z_ = (cv - cv.mean()) / cv.std() if cv.std() > 0 else cv * 0
            else:
                sd = cv.std(axis=1, keepdims=True)
                z_ = np.where(sd > 0, (cv - cv.mean(axis=1, keepdims=True)) / np.where(sd > 0, sd, 1), 0)
            zb.append(z_[r.c_best]); zs.extend(np.delete(z_, r.c_best, axis=0))
        mb, ms = np.mean(zb, 0), np.mean(zs, 0)
        dts = L.DT_GRID
        print(f"z-scored per {zmode}: best n={len(zb)}, suboptimal n={len(zs)}")
        print(f"   best peaks at {dts[mb.argmax()]} ms, minimum at {dts[mb.argmin()]} ms; suboptimal peaks at {dts[ms.argmax()]} ms, minimum at {dts[ms.argmin()]} ms; "
              f"pooled minimum at {dts[np.mean(zb + zs, 0).argmin()]} ms; suboptimal below best at the best peak: {ms[mb.argmax()] < mb.max()}")
        print("   best mean every 10 ms:", mb[::10].round(2), "\n   suboptimal mean every 10 ms:", ms[::10].round(2))

run_fig7b_inbound_best_outbound_suboptimal(M2)

**Result: partially reproduced.** Z-scored within each unit, the mean best-combination curve of the 318 selective sets peaks at -50 ms and the suboptimal mean (2,226 curves) at +42 ms, below the best mean there. The minimum is at 0 ms for the best mean, -6 ms for the suboptimal mean and -3 ms pooled, none within -10 +- 1 ms; z-scoring each curve gives the same peaks and minima at 0, -5 and -3 ms.

### fig7c-pca

> Of the dt tuning curves, 357 of best (79%) and 1,247 of suboptimal combinations (40%) were significantly modulated; three PCs captured 75% of their variance, PC1 sigmoidal about 0 ms (order) and PC2 U-shaped (speed).

In [ ]:
rows_pc = []
for _, r in M2.iterrows():
    d = CUR[(r.unit, r.set)]
    for c in range(8):
        if d["status"][c] == "modulated":
            cv = d["curves"][c]
            rows_pc.append(dict(unit=r.unit, set=r.set, sel=r.sel, best=c == r.c_best, z=(cv - cv.mean()) / cv.std()))
PC = pd.DataFrame(rows_pc)
Z = np.vstack(PC.z.to_numpy())
nb_, ns_ = int(PC.best.sum()), int((~PC.best).sum())
print(f"modulated best curves {nb_} of {len(M2)} ({nb_ / len(M2):.2f}); suboptimal {ns_} of {7 * len(M2)} ({ns_ / (7 * len(M2)):.2f})")
Zc = Z - Z.mean(0)
u_, s_, vt = np.linalg.svd(Zc, full_matrices=False)
ev = s_ ** 2 / np.sum(s_ ** 2)
print("variance explained by PCs 1-5:", ev[:5].round(3), "; first three:", ev[:3].sum().round(3))
dts = L.DT_GRID
pc1 = vt[0] * (1 if vt[0][dts > 0].mean() > vt[0][dts < 0].mean() else -1)
pc2 = vt[1]
sc1 = Zc @ pc1
PC["pc1"] = sc1; PC["pc2"] = Zc @ pc2; PC["pc3"] = Zc @ vt[2]
zero_cross = dts[np.where(np.diff(np.sign(pc1)) != 0)[0]]
print("PC1 sign changes at dt:", zero_cross, "; PC1 every 10 ms:", pc1[::10].round(3))
far, near = pc2[np.abs(dts) > 30].mean(), pc2[np.abs(dts) < 10].mean()
print(f"PC2 mean |dt| > 30: {far:.3f}, |dt| < 10: {near:.3f} (opposite signs: {np.sign(far) != np.sign(near)}); PC2 every 10 ms:", pc2[::10].round(3))

**Result: partially reproduced.** 373 best (0.81) and 1,204 suboptimal (0.37) curves are dt-modulated, within tolerance of 357 and 1,247. PC1 is sigmoidal with its sign change at 0 ms, and PC2 is U-shaped (0.107 for |dt| > 30 ms, -0.133 for |dt| < 10 ms); but three PCs explain 0.836 of the variance, above the tolerance band (0.675 to 0.825).

### fig7e-pc1-best-suboptimal

> In combination-selective cells, best combinations had negative PC1 weights and suboptimal combinations positive (P = 5.7 x 10-16, t-test), an inbound order preference nearly absent in nonselective cells; PC2 and PC3 weights did not differ across cell or stimulus types.

In [ ]:
def run_fig7e_pc1_best_suboptimal(M2):
    selmap = dict(zip(zip(M2.unit, M2.set), M2.sel))
    PCs = PC.assign(sel=[selmap[(u, s)] for u, s in zip(PC.unit, PC.set)])
    g = PCs.groupby(["sel", "best"]).pc1.agg(["mean", "size"])
    print(g)
    sb, ss = PCs.loc[PCs.sel & PCs.best, "pc1"], PCs.loc[PCs.sel & ~PCs.best, "pc1"]
    nb, ns = PCs.loc[~PCs.sel & PCs.best, "pc1"], PCs.loc[~PCs.sel & ~PCs.best, "pc1"]
    t = stats.ttest_ind(sb, ss)
    print(f"selective best vs suboptimal: t = {t.statistic:.2f}, p = {t.pvalue:.3g}; difference {sb.mean() - ss.mean():.3f}; "
          f"nonselective difference {nb.mean() - ns.mean():.3f} (ratio {(nb.mean() - ns.mean()) / (sb.mean() - ss.mean()):.2f}), p = {stats.ttest_ind(nb, ns).pvalue:.3g}")
    for pc in ["pc2", "pc3"]:
        grps = {"sel best": PCs.loc[PCs.sel & PCs.best, pc], "nonsel best": PCs.loc[~PCs.sel & PCs.best, pc], "all suboptimal": PCs.loc[~PCs.best, pc]}
        ks = list(grps)
        out = [f"{ks[i]} vs {ks[j]}: p = {stats.ttest_ind(grps[ks[i]], grps[ks[j]]).pvalue:.3g}" for i in range(3) for j in range(i + 1, 3)]
        print(pc, {k: round(v.mean(), 3) for k, v in grps.items()}, "|", "; ".join(out))

run_fig7e_pc1_best_suboptimal(M2)

**Result: partially reproduced.** In selective sets, best combinations have negative PC1 weights (-2.37) and suboptimal ones positive (0.21), p = 8.9e-8. The pattern is not nearly absent in nonselective sets: their best-minus-suboptimal difference is 0.59 of the selective one (p = 0.016). For PC2, nonselective best and all suboptimal weights differ (p = 0.005); no PC3 pair differs.

### fig7f-inbound-sharper

> Combination-selective units preferring inbound order (best dt < 0, n = 121) were more sharply tuned than those preferring outbound order (n = 56; P = 0.03) and had better alignment of nonlinear and linear components (P = 0.01).

In [ ]:
def run_fig7f_inbound_sharper(M2):
    sel = M2[M2.sel].copy()
    sel["align_idx"] = [np.corrcoef(r.lin_a, r.resp_a - SLOPE * r.lin_a)[0, 1] for _, r in sel.iterrows()]
    rows = []
    for lab, g in [("(a) dt = 0 excluded", sel[sel.dt_best != 0]), ("(b) dt = 0 with outbound", sel),
                   ("(c)+(a) constant best curve and dt = 0 excluded", sel[~sel.best_const & (sel.dt_best != 0)]),
                   ("(c)+(b) constant best curve excluded", sel[~sel.best_const])]:
        inb, outb = g[g.dt_best < 0], g[g.dt_best >= 0]
        t1, t2 = stats.ttest_ind(inb.csi_a, outb.csi_a), stats.ttest_ind(inb["align_idx"], outb["align_idx"])
        rows.append(dict(reading=lab, inbound=len(inb), outbound=len(outb), csi_in=inb.csi_a.mean(), csi_out=outb.csi_a.mean(), p_csi=t1.pvalue,
                         align_in=inb["align_idx"].mean(), align_out=outb["align_idx"].mean(), p_align=t2.pvalue))
    print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))
    print("selective sets with a constant best-combination curve (best dt falls on -50 ms by the tie):", int(sel.best_const.sum()))

run_fig7f_inbound_sharper(M2)

**Result: partially reproduced.** Excluding the sets whose best curve is constant (reading c), inbound-preferring sets are more sharply tuned (CSI 0.109 against 0.083, p = 0.030 or 0.027) and better aligned (0.319 against 0.038 or 0.033, p = 1e-7). Readings (a) and (b) keep the 70 constant-curve sets, whose best dt falls on -50 ms only by the tie, and so count them as inbound; there alignment holds (p = 3e-4 and 2e-4) and the CSI difference does not (p = 0.22 and 0.21). The group sizes, 134 inbound and 112 or 114 outbound under reading (c), are outside 121 +- 12.1 and 56 +- 5.6.

### Decoders

Per unit (a responsive set), one-versus-all logistic classifiers of combination identity on the single-trial count, trials within +-5 ms of the unit's best dt (all trials for a set whose best curve is constant, which has no best dt), with tenfold cross-validated held-out probabilities; ensembles of N units drawn with replacement, one held-out trial per unit per combination (O6), and the combination with the largest summed probability is the answer. Linear-prediction decoders use counts made by adding a single-trial CW count and a single-trial SW count drawn from the single-whisker trials in the set's directions (Methods, "Neural decoders"). Combinations are identified by SW face position (O5). Sets lacking trials of some combination in the window are dropped (counted below).

In [ ]:
t0 = time.time()
rng_dec = np.random.default_rng(3)
def unit_trials(r, window=None):
    d = CUR[(r.unit, r.set)]
    if window is None:
        m = np.ones(len(d["y"]), bool) if r.best_const else np.abs(d["dti"] - r.j_best) <= 5
    else:
        m = (d["dti"] - 50 >= window[0]) & (d["dti"] - 50 <= window[1])
    return d["y"][m], d["comb"][m]
def lin_trials(r, labels, rng):
    """Linearly predicted single-trial counts: a CW single-trial count plus a SW single-trial count."""
    d = CUR[(r.unit, r.set)]
    tab = T2[r.recording]
    sing = tab[tab.kind == "single"]
    x = np.zeros(len(labels))
    cwv = sing.loc[(sing.a == L.CW12) & sing.da.isin(list(d["dirs"][L.CW12])), f"c{r.unit}"].to_numpy()
    for k in range(8):
        w = L.SWS12[k]
        swv = sing.loc[(sing.a == w) & sing.da.isin(list(d["dirs"][w])), f"c{r.unit}"].to_numpy()
        ii = np.where(labels == k)[0]
        x[ii] = rng.choice(cwv, len(ii)) + rng.choice(swv, len(ii))
    return x
DEC, DECL, dropped = {}, {}, 0
for i, r in M2.iterrows():
    x, y = unit_trials(r)
    if len(np.unique(y)) < 8:
        dropped += 1
        continue
    DEC[i] = (L.heldout_prob_rows(x, y, 8, rng_dec), y)
    xl = lin_trials(r, y, rng_dec)
    DECL[i] = (L.heldout_prob_rows(xl, y, 8, rng_dec), y)
print(f"classifiers for {len(DEC)} sets ({dropped} dropped) in {time.time() - t0:.0f} s")
ids = np.array(list(DEC))
POOL = L.Pool([DEC[i][0] for i in ids], [DEC[i][1] for i in ids], 8)
POOLL = L.Pool([DECL[i][0] for i in ids], [DECL[i][1] for i in ids], 8)
pos_of = {i: j for j, i in enumerate(ids)}
def members(mask):
    return np.array([pos_of[i] for i in M2.index[mask] if i in pos_of])
SEL_IDX = members(M2.sel.to_numpy())

### fig3e-identity-decoder

> A population decoder of CW-SW combination identity from single-trial counts reached 42% accuracy for 187 selective units (chance 12.5%), 82% for the rostral within-row SW and 23 to 28% for the three dorsal-row SWs, and was weaker on linearly predicted counts.

In [ ]:
def run_fig3e_identity_decoder(M2):
    SEL_IDX = members(M2.sel.to_numpy())
    rng_3e = np.random.default_rng(31)
    names = [L.POSNAME[L.relpos(w, 2)] for w in L.SWS12]
    for N in sorted({10, 50, 100, 187, len(SEL_IDX)}):
        acc = POOL.accuracy(N, 2500, rng_3e, SEL_IDX)
        accl = POOLL.accuracy(N, 2500, rng_3e, SEL_IDX)
        m, ml = acc.mean(1), accl.mean(1)
        print(f"N = {N}: measured {m.mean():.3f} (s.d. {m.std():.3f}); linear prediction {ml.mean():.3f} (s.d. {ml.std():.3f}); "
              f"difference {m.mean() - ml.mean():.3f} vs sum of s.d. {m.std() + ml.std():.3f}")
        if N == 187:
            print("   per combination at N = 187:", dict(zip(names, acc.mean(0).round(3))))
    print("selective sets in the pool:", len(SEL_IDX), "; chance 0.125")
    single = np.array([np.mean(np.argmax(DEC[i][0], 1) == DEC[i][1]) for i in ids[SEL_IDX]])
    print(f"single-unit held-out accuracy (own argmax), selective sets: mean {single.mean():.3f}, range {single.min():.3f} to {single.max():.3f}")
    print("best combination of the selective sets by face position:", pd.Series(np.array(names)[M2.loc[M2.sel, "c_best"]]).value_counts().to_dict())

run_fig3e_identity_decoder(M2)

**Result: not reproduced.** The summed-probability decoder reaches 0.145 at N = 187 selective sets (chance 0.125), against 0.42; it reports the rostral within-row combination for nearly every stimulus (accuracy 0.987 for that combination, 0 to 0.059 for the others, against 0.82 and 0.23 to 0.28 for the dorsal row). The linear-prediction decoder is lower by 0.013, less than the sum of the two s.d. (0.096).

### fig3g-binary-decoder

> A binary decoder discriminating each unit's best CW-SW combination from suboptimal combinations reached 70% accuracy for 10 neurons and over 90% for 100.

In [ ]:
def binary_pool(rows_, mode, rng):
    probs, labs = [], []
    for i, r in rows_.iterrows():
        x, y = unit_trials(r)
        best = y == r.c_best
        if mode == "a":  # suboptimal trials pooled over the seven, as many as best trials
            sub = rng.choice(np.where(~best)[0], best.sum(), replace=best.sum() > (~best).sum())
        else:  # one suboptimal combination drawn at random
            c2 = rng.choice([c for c in range(8) if c != r.c_best])
            sub = np.where(y == c2)[0]
        ii = np.r_[np.where(best)[0], sub]
        lab = np.r_[np.ones(best.sum(), int), np.zeros(len(sub), int)]
        P = L.heldout_prob_rows(x[ii], lab, 2, rng)
        probs.append(P); labs.append(lab)
    return L.Pool(probs, labs, 2)
def run_fig3g_binary_decoder(M2):
    rng_3g = np.random.default_rng(32)
    rows = []
    for units_lab, rows_ in [("(c) selective", M2[M2.sel]), ("(d) all", M2)]:
        for mode in "ab":
            r_ = dict(units=units_lab, suboptimal=mode, n_units=len(rows_))
            reps = {10: [], 100: []}
            for rep_ in range(3):  # three refits of the held-out probabilities, 10,000 ensembles each
                pool = binary_pool(rows_, mode, rng_3g)
                for N in [10, 100]:
                    reps[N].append(pool.accuracy(N, 10000, rng_3g).mean(1).mean())
            for N in [10, 100]:
                r_[f"N{N}"] = np.mean(reps[N])
                r_[f"N{N}_sd_over_refits"] = np.std(reps[N], ddof=1)
            rows.append(r_)
    print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))
    print("Monte Carlo s.e. of one mean over 10,000 ensembles is about s.d. / 100 (the s.d. over ensembles is about 0.3 at N = 10)")

run_fig3g_binary_decoder(M2)

**Result: partially reproduced.** Averaged over three refits of the held-out probabilities with 10,000 ensembles each, the best-against-suboptimal decoder over the selective sets reaches 0.633 (reading a) and 0.628 (reading b) at N = 10, and 0.857 and 0.832 at N = 100; over all sets 0.607 and 0.601 at N = 10. Under readings (a) and (c), accuracy at N = 10 is within the tolerance band (0.63 to 0.77) and accuracy at N = 100 is above 0.81 but not above 0.9. The N = 10 value is 0.003 above the band's edge, with an s.d. over refits of 0.010, so the outcome is fragile; over the 187 sets with the smallest p (variant section) the decoder reaches 0.670 and 0.914, and over the 162 calibrated sets 0.681 and 0.929. With 20 further refits (band-edge checks) the N = 10 mean is 0.6329 with an s.e. of 0.0012, 2.4 s.e. above the edge.

### fig4c-sw-tuned-decoding

> Combination identity was decoded from SW-tuned units more accurately and efficiently than from CW-tuned units, and decoding from CW-tuned units was still above chance.

In [ ]:
def run_fig4c_sw_tuned_decoding(M2):
    rng_4c = np.random.default_rng(4)
    rows = []
    for lab, base in [("(a) selective", M2.sel.to_numpy()), ("(b) all", np.ones(len(M2), bool))]:
        for grp, gm in [("SW-tuned", ~M2.cw_tuned_a.to_numpy()), ("CW-tuned", M2.cw_tuned_a.to_numpy())]:
            idx = members(base & gm)
            r_ = dict(units=lab, group=grp, n=len(idx))
            for N in [10, 25, 50, 100, 150]:
                acc = POOL.accuracy(N, 2500, rng_4c, idx).mean(1)
                r_[f"N{N}"] = acc.mean()
                if N == 150:
                    r_["sd150"] = acc.std()
            rows.append(r_)
    print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

run_fig4c_sw_tuned_decoding(M2)

**Result: partially reproduced.** Over all sets (reading b), SW-tuned decoders are more accurate than CW-tuned ones at every size (0.139 against 0.116 at N = 10, 0.146 against 0.130 at N = 150); over selective sets (reading a) they are more accurate up to N = 100 and not at N = 150 (0.151 against 0.156). CW-tuned decoders at N = 150 (0.130 and 0.156) are not above chance (0.125) by two s.d. (s.d. 0.055 and 0.075).

### fig7g-inbound-decoding

> Spatial combination decoding was better for inbound-order stimuli than for outbound-order stimuli.

In [ ]:
RANGES = {"claims file": [(-50, -41), (-40, -31), (-30, -21), (-20, -11), (-10, -1), (0, 9), (10, 19), (20, 29), (30, 39), (41, 50)],
          "mirror images of the negative ranges (variant)": [(-50, -41), (-40, -31), (-30, -21), (-20, -11), (-10, -1), (1, 10), (11, 20), (21, 30), (31, 40), (41, 50)]}
def range_decoding(rows_, ranges, rng):
    out = {}
    for rg in ranges:
        probs, labs, keep = [], [], []
        for i, r in rows_.iterrows():
            x, y = unit_trials(r, window=rg)
            if len(np.unique(y)) < 8:
                continue
            probs.append(L.heldout_prob_rows(x, y, 8, rng)); labs.append(y); keep.append(i)
        pool = L.Pool(probs, labs, 8)
        cw = rows_.loc[np.array(keep), "cw_tuned_a"].to_numpy()
        res = {}
        for lab, N, units in [("(a) N=187", 187, None), ("(b) N=100", 100, None),
                              ("(c) CW-tuned N=100", 100, np.where(cw)[0]), ("(c) SW-tuned N=100", 100, np.where(~cw)[0])]:
            acc = pool.accuracy(N, 1000, rng, units).mean(1)
            res[lab] = (acc.mean(), acc.std())
        out[rg] = res
    return out
def run_fig7g_inbound_decoding(M2):
    rng_7g = np.random.default_rng(7)
    for rname, ranges in RANGES.items():
        out7 = range_decoding(M2[M2.sel], ranges, rng_7g)
        tab7 = pd.DataFrame({str(rg): {k: v[0] for k, v in res.items()} for rg, res in out7.items()}).T
        print(f"ranges: {rname}")
        print(tab7.round(3).to_string())
        for col in tab7.columns:
            neg, pos = tab7[col].iloc[:5].mean(), tab7[col].iloc[5:].mean()
            sd = np.mean([out7[rg][col][1] for rg in ranges])
            print(f"   {col}: mean over the five negative ranges {neg:.3f}, over the five others {pos:.3f}, difference {neg - pos:.3f}, mean bootstrap s.d. {sd:.3f}")

run_fig7g_inbound_decoding(M2)

**Result: not reproduced.** Decoding over the five negative dt ranges exceeds that over the five others by 0.014 (N = 187), 0.016 (N = 100), 0.014 (CW-tuned) and 0.019 (SW-tuned), each smaller than the mean bootstrap s.d. (0.057 to 0.075); accuracies stay between 0.094 and 0.165 (chance 0.125). With the positive ranges set as mirror images of the negative ones (variant), the differences are 0.009 to 0.015, against s.d. 0.059 to 0.077.

### supp8-order-decoding

> A population decoder of sequence order (dt in -50 to -10 against +10 to +50 ms) from best-combination responses was about 65% correct for 100 units, against chance of 50%.

In [ ]:
def order_pool(rows_, rng):
    probs, labs = [], []
    for i, r in rows_.iterrows():
        d = CUR[(r.unit, r.set)]
        dt_ = d["dti"] - 50
        m = (d["comb"] == r.c_best) & (np.abs(dt_) >= 10)
        lab_ = (dt_[m] > 0).astype(int)
        probs.append(L.heldout_prob_rows(d["y"][m], lab_, 2, rng)); labs.append(lab_)
    return L.Pool(probs, labs, 2)
def run_supp8_order_decoding(M2):
    rng_s8 = np.random.default_rng(8)
    for lab, rows_ in [("(a) selective", M2[M2.sel]), ("(b) all", M2)]:
        res = {10: [], 100: []}
        sds = []
        for rep_ in range(5):  # five refits of the held-out probabilities, 10,000 ensembles each
            pool = order_pool(rows_, rng_s8)
            for N in [10, 100]:
                acc = pool.accuracy(N, 10000, rng_s8).mean(1)
                res[N].append(acc.mean())
                if N == 100:
                    sds.append(acc.std())
        for N in [10, 100]:
            print(f"{lab}, N = {N}: accuracy {np.mean(res[N]):.3f} over five refits (range {min(res[N]):.3f} to {max(res[N]):.3f}, s.d. {np.std(res[N], ddof=1):.4f}); chance 0.5")
        print(f"   N = 100: s.d. over ensembles {np.mean(sds):.3f}, so the Monte Carlo s.e. of one 10,000-ensemble mean is {np.mean(sds) / 100:.4f}")
    print(f"decoders: {time.time() - t0:.0f} s")

run_supp8_order_decoding(M2)

**Result: not reproduced.** Averaged over five refits with 10,000 ensembles each, the order decoder reaches 0.584 at N = 100 over selective sets (refits 0.580 to 0.589) and 0.571 over all sets, below the tolerance band (0.585 to 0.715); chance is 0.5. The selective value is 0.001 below the band, less than the s.d. over refits (0.004), so the outcome is fragile; over the 187 sets with the smallest p (variant section) the decoder reaches 0.595 and over the 162 calibrated sets 0.598, both within the band. With 20 further refits (band-edge checks) the selective mean is 0.5840 with an s.e. of 0.0008, 1.2 s.e. below the edge, so the expected accuracy is within about 0.002 of it.

## Selective-group claims on a group of the paper's size

The shuffle test finds 318 selective sets where the paper found 187, and the real-data null above shows that it rejects far more than 5% of sets with no combination tuning. Every claim on the selective group is therefore run again here on the 187 sets with the smallest shuffle p-values (`M2v`; the nonselective group is the other 276), to show which outcomes depend on the inflated group. The code is that of each claim's section above.

In [ ]:
print("fig2f-shuffle-abolishes-tuning, the 187 sets with the smallest p:")
run_fig2f_shuffle_abolishes_tuning(M2v)

In [ ]:
print("fig2f-most-selective-lower-firing, the 187 sets with the smallest p:")
run_fig2f_most_selective_lower_firing(M2v)

In [ ]:
print("supp4-selectivity-at-high-rates, the 187 sets with the smallest p:")
run_supp4_selectivity_at_high_rates(M2v)

In [ ]:
print("fig3b-sharper-in-both-groups, the 187 sets with the smallest p:")
run_fig3b_sharper_in_both_groups(M2v)

In [ ]:
print("fig3b-linear-sharper-in-selective, the 187 sets with the smallest p:")
run_fig3b_linear_sharper_in_selective(M2v)

In [ ]:
print("supp6-layers, the 187 sets with the smallest p:")
run_supp6_layers(M2v)

In [ ]:
print("supralinear-best-8-percent, the 187 sets with the smallest p:")
run_supralinear_best_8_percent(M2v)

In [ ]:
print("fig6b-best-above-worst-below-scaled, the 187 sets with the smallest p:")
run_fig6b_best_above_worst_below_scaled(M2v)

In [ ]:
print("fig6b-enhancement-not-different, the 187 sets with the smallest p:")
run_fig6b_enhancement_not_different(M2v)

In [ ]:
print("fig6b-suppression-greater-selective, the 187 sets with the smallest p:")
run_fig6b_suppression_greater_selective(M2v)

In [ ]:
print("fig6c-enhancement-strongest-sw, the 187 sets with the smallest p:")
run_fig6c_enhancement_strongest_sw(M2v)

In [ ]:
print("fig7a-dt-half-width, the 187 sets with the smallest p:")
run_fig7a_dt_half_width(M2v)

In [ ]:
print("fig7b-inbound-best-outbound-suboptimal, the 187 sets with the smallest p:")
run_fig7b_inbound_best_outbound_suboptimal(M2v)

In [ ]:
print("fig7e-pc1-best-suboptimal, the 187 sets with the smallest p:")
run_fig7e_pc1_best_suboptimal(M2v)

In [ ]:
print("fig7f-inbound-sharper, the 187 sets with the smallest p:")
run_fig7f_inbound_sharper(M2v)

In [ ]:
print("fig3e-identity-decoder, the 187 sets with the smallest p:")
run_fig3e_identity_decoder(M2v)

In [ ]:
print("fig3g-binary-decoder, the 187 sets with the smallest p:")
run_fig3g_binary_decoder(M2v)

In [ ]:
print("fig4c-sw-tuned-decoding, the 187 sets with the smallest p:")
run_fig4c_sw_tuned_decoding(M2v)

In [ ]:
print("fig7g-inbound-decoding, the 187 sets with the smallest p:")
run_fig7g_inbound_decoding(M2v)

In [ ]:
print("supp8-order-decoding, the 187 sets with the smallest p:")
run_supp8_order_decoding(M2v)

**Result of the variant section.** On the 187 sets with the smallest shuffle p, compared with the outcomes above on the 318 sets:

- Same outcome: fig2f-shuffle-abolishes-tuning (CSI 0.039 against 0.000; first shuffled point 0.020 above the others, s.e.m. 0.037), fig2f-most-selective-lower-firing (p = 3e-9 and 1e-5), fig3b-sharper-in-both-groups (p = 3e-12 and 7e-14), fig3b-linear-sharper-in-selective (p = 1.4e-5), supp6-layers (selective sets p = 0.0043, L6 count unchanged), supralinear-best-8-percent (0.080 under O3 b), fig6b-best-above-worst-below-scaled (all p < 1e-14), fig6b-suppression-greater-selective (Bonferroni p = 0.0002 and 0.0001), fig6c-enhancement-strongest-sw (selective p = 7e-15, nonselective p = 0.21), fig7b-inbound-best-outbound-suboptimal (best peak -44 ms, suboptimal +42 ms, minima 2, -5 and 0 ms), fig7e-pc1-best-suboptimal (p = 1.2e-8; nonselective ratio 0.37; PC2 nonselective best against suboptimal p = 0.002), fig3e-identity-decoder (0.148 at N = 187), fig4c-sw-tuned-decoding (reading a: SW-tuned higher at every size, 0.163 against 0.140 at N = 150; CW-tuned 0.140 with s.d. 0.050) and fig7g-inbound-decoding (differences 0.004 to 0.015 against s.d. 0.034 to 0.068).
- fig6b-enhancement-not-different is not reproduced under O3 (a) (p = 0.014) and would be reproduced under O3 (b) (p = 0.107), where on the 318 sets both readings give p < 0.05.
- fig7f-inbound-sharper stays partial for another reason: under reading (c) alignment differs (p = 0.033 with dt = 0 counted outbound) and CSI does not (p = 0.51), with 92 inbound and 41 or 42 outbound sets.
- Different outcome: supp4-selectivity-at-high-rates (O3 b: 88, 99 and 82, all within tolerance, peak 17.3 times its s.e.m.; reproduced), fig7a-dt-half-width (13.0 ms over the 134 dt-modulated sets, outside 10 +- 2, and a suboptimal half-width of 4.5 ms; not reproduced), fig3g-binary-decoder (0.670 at N = 10 and 0.914 at N = 100, reading a; reproduced) and supp8-order-decoding (0.595 at N = 100, within the band; reproduced).

The outcomes of these four claims, and of fig6b-enhancement-not-different, depend on which selective group is used, and are marked as resting on the assumption that the selective group is the one the shuffle test of the Methods gives. The next section runs the same claims on the calibrated group.

## Selective-group claims on the calibrated group

The round-2 verifier asked for the selective-group claims on the group that the calibration gives, which does not fix its size at the paper's. Here the selective group is the 162 sets selected by FDR on the p-values calibrated on the real-data null (`M2c`, see exp2-combination-selective-prevalence); the nonselective group is the other 301. The code is that of each claim's section above.

In [ ]:
print("fig2f-shuffle-abolishes-tuning, the calibrated group of 162 sets:")
run_fig2f_shuffle_abolishes_tuning(M2c)

In [ ]:
print("fig2f-most-selective-lower-firing, the calibrated group of 162 sets:")
run_fig2f_most_selective_lower_firing(M2c)

In [ ]:
print("supp4-selectivity-at-high-rates, the calibrated group of 162 sets:")
run_supp4_selectivity_at_high_rates(M2c)

In [ ]:
print("fig3b-sharper-in-both-groups, the calibrated group of 162 sets:")
run_fig3b_sharper_in_both_groups(M2c)

In [ ]:
print("fig3b-linear-sharper-in-selective, the calibrated group of 162 sets:")
run_fig3b_linear_sharper_in_selective(M2c)

In [ ]:
print("supp6-layers, the calibrated group of 162 sets:")
run_supp6_layers(M2c)

In [ ]:
print("supralinear-best-8-percent, the calibrated group of 162 sets:")
run_supralinear_best_8_percent(M2c)

In [ ]:
print("fig6b-best-above-worst-below-scaled, the calibrated group of 162 sets:")
run_fig6b_best_above_worst_below_scaled(M2c)

In [ ]:
print("fig6b-enhancement-not-different, the calibrated group of 162 sets:")
run_fig6b_enhancement_not_different(M2c)

In [ ]:
print("fig6b-suppression-greater-selective, the calibrated group of 162 sets:")
run_fig6b_suppression_greater_selective(M2c)

In [ ]:
print("fig6c-enhancement-strongest-sw, the calibrated group of 162 sets:")
run_fig6c_enhancement_strongest_sw(M2c)

In [ ]:
print("fig7a-dt-half-width, the calibrated group of 162 sets:")
run_fig7a_dt_half_width(M2c)

In [ ]:
print("fig7b-inbound-best-outbound-suboptimal, the calibrated group of 162 sets:")
run_fig7b_inbound_best_outbound_suboptimal(M2c)

In [ ]:
print("fig7e-pc1-best-suboptimal, the calibrated group of 162 sets:")
run_fig7e_pc1_best_suboptimal(M2c)

In [ ]:
print("fig7f-inbound-sharper, the calibrated group of 162 sets:")
run_fig7f_inbound_sharper(M2c)

In [ ]:
print("fig3e-identity-decoder, the calibrated group of 162 sets:")
run_fig3e_identity_decoder(M2c)

In [ ]:
print("fig3g-binary-decoder, the calibrated group of 162 sets:")
run_fig3g_binary_decoder(M2c)

In [ ]:
print("fig4c-sw-tuned-decoding, the calibrated group of 162 sets:")
run_fig4c_sw_tuned_decoding(M2c)

In [ ]:
print("fig7g-inbound-decoding, the calibrated group of 162 sets:")
run_fig7g_inbound_decoding(M2c)

In [ ]:
print("supp8-order-decoding, the calibrated group of 162 sets:")
run_supp8_order_decoding(M2c)

**Result of the calibrated-group section.** The 162 calibrated sets are all among the 318 of the main analysis; they lie in 111 cells and are 68 correlated and 94 anti-correlated sets. Compared with the outcomes above on the 318 sets:

- Same outcome: fig2f-shuffle-abolishes-tuning (CSI 0.041 against 0.000; first shuffled point 0.009 above the others, s.e.m. 0.039), fig2f-most-selective-lower-firing (p = 8e-9 and 3e-5), fig3b-sharper-in-both-groups (p = 2e-10 and 1e-15), fig3b-linear-sharper-in-selective (p = 4.4e-7), supp6-layers (selective sets p = 0.014, L2/3 and L4 above L5 and L6; L6 count unchanged), fig6b-best-above-worst-below-scaled (all p < 1e-15), fig6b-suppression-greater-selective (Bonferroni over eight p = 0.0009 and 0.0001), fig6c-enhancement-strongest-sw (selective p = 1e-15, nonselective p = 0.35), fig7b-inbound-best-outbound-suboptimal (best peak -44 ms, suboptimal +42 ms, minima 2, -6 and 1 ms), fig7e-pc1-best-suboptimal (p = 1.6e-7, nonselective ratio 0.43; the selective suboptimal mean is negative, -0.65, as it also is on the 187 sets), fig7f-inbound-sharper (under reading c alignment differs, p = 0.019, and CSI does not, p = 0.26), fig3e-identity-decoder (0.167 at N = 162), fig4c-sw-tuned-decoding (reading a: CW-tuned above SW-tuned at N = 100 and 150, 0.195 against 0.167 with 27 CW-tuned sets; reading b as above), fig7g-inbound-decoding (differences -0.001 to 0.018 against s.d. 0.058 to 0.076), supp4-selectivity-at-high-rates (74, 88 and 80 under O3 a, 74, 88 and 93 under O3 b, so the first count misses 95 +- 9.5; peak 19.7 times its s.e.m.; partially reproduced) and fig7a-dt-half-width (12.0 ms over the 114 dt-modulated sets, at the edge of 10 +- 2, suboptimal lower at the best dt, 0.580, but with a narrower half-width of 7.0 ms; partially reproduced).
- fig6b-enhancement-not-different is not reproduced under O3 (a) (p = 0.0125) and would be reproduced under O3 (b) (p = 0.073), as on the 187 sets.
- Different outcome: fig3g-binary-decoder (0.681 at N = 10 and 0.929 at N = 100, reading a; reproduced), supp8-order-decoding (0.598 at N = 100, within the band; reproduced) and supralinear-best-8-percent (0.031 of selective sets under O3 a and 0.093 under O3 b, and 0.093 and 0.151 of all sets, all outside 0.072 to 0.088; not reproduced).

So fig3g and supp8 would be reproduced on either smaller group, and supp4 and fig7a only on the 187 sets. supralinear-best-8-percent is reproduced on the 318 sets (O3 a) and the 187 sets (O3 b) and not on the 162 calibrated sets, so its outcome also rests on the selective group.

## Band-edge checks with more refits

Two decoder outcomes lie within one s.d. over refits of a band edge: fig3g-binary-decoder (0.633 at N = 10 against a lower bound of 0.63) and supp8-order-decoding (0.584 at N = 100 against 0.585). Each is run here again on the selective sets with 20 refits of the held-out probabilities (new random draws, 10,000 ensembles each), so that the mean has a standard error near 0.002. fig3g uses reading (a) at N = 10; supp8 uses N = 100.

In [ ]:
rng_edge = np.random.default_rng(2020)
for claim, make, N, edge in [("fig3g-binary-decoder, reading (a), N = 10", lambda: binary_pool(M2[M2.sel], "a", rng_edge), 10, 0.63),
                             ("supp8-order-decoding, N = 100", lambda: order_pool(M2[M2.sel], rng_edge), 100, 0.585)]:
    reps = np.array([make().accuracy(N, 10000, rng_edge).mean(1).mean() for _ in range(20)])
    se = reps.std(ddof=1) / np.sqrt(len(reps))
    print(f"{claim}: mean over 20 refits {reps.mean():.4f} (s.e. {se:.4f}, range {reps.min():.3f} to {reps.max():.3f}); "
          f"band edge {edge}, difference {reps.mean() - edge:+.4f} = {(reps.mean() - edge) / se:+.1f} s.e.")

**Result of the band-edge checks.** With 20 refits, fig3g-binary-decoder reaches 0.6329 at N = 10 (s.e. 0.0012), 2.4 s.e. above the band's lower edge of 0.63, so its partial outcome holds within the Monte Carlo error. supp8-order-decoding reaches 0.5840 at N = 100 (s.e. 0.0008), 1.2 s.e. below the band's lower edge of 0.585: its expected accuracy lies within about 0.002 of the edge, and the outcome (not reproduced) would flip with a change of that size. supralinear-best-8-percent is a count, 23 of 318 sets (0.0723), with no Monte Carlo error; one set fewer would put it below the band.

## Experiment 3 (awake mice)

Epochs are aligned to the first deflection (data check). The files hold no lick times and no Go or No-go labels, so the exclusion of epochs within 200 ms of a lick (Methods) cannot be applied: every epoch is used. A variant drops, per unit, the epochs whose spike list holds a time after 300 ms (see the data check; what these are is not documented). Sham reading (a) is the 0 to 125 ms window of sham epochs; (b) the whole 300 ms sham epoch, scaled.

In [ ]:
def exp3_table(U, T, drop_late=False):
    rows, seqs = [], {}
    for rec, tab in T.items():
        for k in U.loc[U.recording == rec, "unit"]:
            t_ = tab[tab[f"o{k}"] == 0] if drop_late else tab
            r = dict(unit=k, recording=rec)
            for sr in "ab":
                r[f"p_resp_{sr}"] = L.lrt_responsive(t_, k, (t_.kind == "single").to_numpy(), sr, 300.0)[1]
                r[f"spont_{sr}"] = L.spont_rate(t_, k, sr, 300.0)
            sing = t_[t_.kind == "single"].groupby("a")[f"c{k}"].mean()
            for w in range(1, 10):
                r[f"s{w}"] = sing.loc[w]
            rows.append(r)
            seqs[k] = t_[t_.kind.isin(["cwsw", "swsw"])].groupby(["kind", "a", "b", "dt"])[f"c{k}"].mean()
    df = U[["unit", "file", "recording", "animal"]].merge(pd.DataFrame(rows), on=["unit", "recording"])
    return df, seqs
E3, SEQ3 = exp3_table(U3, T3)
E3d, SEQ3d = exp3_table(U3, T3, drop_late=True)
W = [f"s{w}" for w in range(1, 10)]

### exp3-units-cw-tuned

> In awake mice (experiment 3), 81 single units were recorded, 74 were whisker-responsive and 62% (46) of these were CW-tuned; average single-whisker tuning peaked at the CW.

In [ ]:
for lab, df in [("all epochs", E3), ("epochs with late spikes dropped", E3d)]:
    for sr in "ab":
        r = df[df[f"p_resp_{sr}"] < 0.05]
        cw = (r[W].to_numpy().argmax(1) + 1 == L.CW3)
        mean_s = r[W].mean()
        print(f"{lab}, sham {sr}: units {len(df)}, responsive {len(r)}, CW-tuned {cw.sum()} ({cw.mean():.2f}); "
              f"largest mean single-whisker response: whisker {int(mean_s.values.argmax()) + 1} (CW = {L.CW3})")
R3 = E3[E3.p_resp_a < 0.05].copy()
R3["cw_tuned"] = R3[W].to_numpy().argmax(1) + 1 == L.CW3
print("mean single-whisker response by face position (responsive, all epochs, sham a):",
      {L.POSNAME[L.relpos(w, 3)]: round(R3[f"s{w}"].mean(), 3) for w in range(1, 10)})
print("responsive per mouse:", R3.groupby("animal").size().to_dict(), "of", E3.groupby("animal").size().to_dict())

**Result: reproduced.** Of 81 units, 74 are responsive and 46 (0.62) CW-tuned under sham reading (a), and the mean single-whisker response is largest for the CW (0.845 spikes per stimulus); sham reading (b) and the variant without late-spike epochs give 75 to 78 and 46 to 47. The lick exclusion could not be applied.

### fig8c-awake-cw-stronger

> Among CW-tuned units, CW-evoked spiking was about twofold stronger, and sharper, in awake mice than in anesthetized mice of experiment 1 (P < 0.001, two-sample t-test).

In [ ]:
for sr in "ab":
    aw = R3[R3.cw_tuned]
    an = R1[R1.cw_tuned]
    cw_aw = aw[f"s{L.CW3}"] - aw[f"spont_{sr}"] * L.WINDOW / 1000
    cw_an = an[f"s{L.CW12}"] - an[f"spont_{sr}"] * L.WINDOW / 1000
    t = stats.ttest_ind(cw_aw, cw_an)
    sw_aw = np.sort(aw[[f"s{w}" for w in range(1, 10) if w != L.CW3]].to_numpy(), 1)[:, ::-1] / aw[[f"s{L.CW3}"]].to_numpy()
    sw_an = np.sort(an[[f"s{w}" for w in range(1, 10) if w != L.CW12]].to_numpy(), 1)[:, ::-1] / an[[f"s{L.CW12}"]].to_numpy()
    print(f"sham {sr}: CW above spontaneous, awake {cw_aw.mean():.3f} (n={len(aw)}), anesthetized {cw_an.mean():.3f} (n={len(an)}), "
          f"ratio {cw_aw.mean() / cw_an.mean():.2f}, t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
print("ranked SW responses / CW, awake:", sw_aw.mean(0).round(3), "mean", sw_aw.mean().round(3))
print("ranked SW responses / CW, anesthetized:", sw_an.mean(0).round(3), "mean", sw_an.mean().round(3))

**Result: not reproduced.** The CW response above spontaneous is 0.709 spikes per stimulus in 46 awake CW-tuned units and 0.414 in 87 anesthetized ones, a ratio of 1.71, within tolerance of 2 +- 0.3, and awake tuning is sharper (mean SW/CW ratio 0.375 against 0.393). The two-sample t-test gives p = 0.0021 (sham b 0.0025), not below 0.001. The criterion's partial rule needs the test to hold, so the outcome is not reproduced, with the ratio and the sharper tuning reproduced.

### fig8d-awake-best-sequence

> In awake mice the best two-whisker sequence evoked more spikes than the best single whisker in nearly all units (slope 1.24).

In [ ]:
x = R3[W].max(1).to_numpy(); y = np.array([SEQ3[k].max() for k in R3.unit])
lr = stats.linregress(x, y)
print(f"(a) slope through origin {(x @ y) / (x @ x):.3f}; (b) with intercept {lr.slope:.3f} (intercept {lr.intercept:.3f}); "
      f"best sequence > best single in {np.sum(y > x)} of {len(x)} ({np.mean(y > x):.3f}); sequences per unit {len(SEQ3[R3.unit.iloc[0]])}")
print("by mouse, fraction above:", R3.assign(above=y > x).groupby("animal").above.mean().round(2).to_dict())

**Result: reproduced.** The slope with an intercept is 1.234 (reading b, within 1.24 +- 0.124; 1.300 through the origin), and 73 of 74 units (0.986) respond more to their best sequence.

### exp3-combination-selective

> In awake mice, 39% of responsive units were significantly combination-selective (29 units); 76% of these and 80% of all responsive units preferred a sequence containing the CW.

In [ ]:
rng_e3 = np.random.default_rng(83)
SW3 = [w for w in range(1, 10) if w != L.CW3]
p3, X3 = [], {}
for _, r in R3.iterrows():
    tab = T3[r.recording]
    st = tab[tab.kind == "cwsw"]
    y = st[f"c{r.unit}"].to_numpy(float)
    comb = st.b.map({w: i for i, w in enumerate(SW3)}).to_numpy()
    dti = st.dt.map({-40.0: 0, 0.0: 1, 40.0: 2}).to_numpy()
    N = np.bincount(comb * 3 + dti, minlength=24).reshape(8, 3)
    meas = np.bincount(comb * 3 + dti, weights=y, minlength=24).reshape(8, 3) / N
    csi = L.sparseness(meas.T)
    order = np.argsort(dti, kind="stable")
    ys, cs, ds = y[order], comb[order], dti[order]
    keys = ds[None, :] + rng_e3.random((5000, len(y)))
    cperm = cs[np.argsort(keys, axis=1)]
    idx = (np.arange(5000)[:, None] * 24 + cperm * 3 + ds[None, :]).ravel()
    S = np.bincount(idx, weights=np.tile(ys, 5000), minlength=5000 * 24).reshape(5000, 8, 3) / N[None]
    sh = L.sparseness(np.transpose(S, (0, 2, 1)))
    p, _ = L.mahalanobis_p(csi, sh)
    p3.append(p)
    X3[r.unit] = dict(meas=meas, y=y, comb=comb, dti=dti)
R3["p_shuffle"] = p3
R3["sel"] = L.bh(R3.p_shuffle.to_numpy())
def best_has_cw(k):
    s = SEQ3[k]
    comb = s.groupby(level=["kind", "a", "b"]).max()
    return comb.idxmax()[0] == "cwsw"
R3["best_cw"] = [best_has_cw(k) for k in R3.unit]
print(f"selective (FDR 0.05): {R3.sel.sum()} of {len(R3)} ({R3.sel.mean():.3f}); uncorrected p < 0.05: {(R3.p_shuffle < 0.05).sum()}")
print(f"best sequence contains the CW: selective {R3.loc[R3.sel, 'best_cw'].mean():.3f}, all responsive {R3.best_cw.mean():.3f}")
print("selective per mouse:", R3[R3.sel].groupby("animal").size().to_dict())

**Result: reproduced.** 27 of 74 responsive units (0.365) are combination-selective after FDR control, within tolerance of 29 and 0.39; the best sequence contains the CW for 0.704 of selective and 0.784 of all responsive units, within tolerance of 0.76 and 0.80.

### fig8h-awake-linearity, fig8i-awake-sharper-than-linear, fig8j-awake-inbound-stronger

> In awake mice, responses to each unit's best CW-SW combination were roughly linear (linearity index 0.93 +- 0.03) and to suboptimal combinations sublinear (0.71 +- 0.009, P = 6.2 x 10-17, two-sample t-test).

> In awake mice, combination tuning was sharper than linearly predicted for virtually all responsive units (P = 9.6 x 10-11, paired t-test).

> In awake mice, responses to inbound CW-SW sequences (dt = -40 ms) were stronger than to outbound (+40 ms) or simultaneous (0 ms) deflections (P = 2.2 x 10-14, one-factor ANOVA).

In [ ]:
LI_b, LI_s, CSI_m, CSI_l = [], [], [], []
for _, r in R3.iterrows():
    meas = X3[r.unit]["meas"]
    c, j = np.unravel_index(np.argmax(meas), meas.shape)
    lin = np.array([r[f"s{L.CW3}"] + r[f"s{w}"] for w in SW3])
    li = meas[:, j] / lin
    LI_b.append(li[c]); LI_s.extend(np.delete(li, c))
    CSI_m.append(L.sparseness(meas[:, j])); CSI_l.append(L.sparseness(lin))
LI_b, LI_s, CSI_m, CSI_l = map(np.array, (LI_b, LI_s, CSI_m, CSI_l))
t = stats.ttest_ind(LI_b, LI_s)
print(f"linearity index: best {LI_b.mean():.3f} +- {stats.sem(LI_b):.3f} (n={len(LI_b)}), suboptimal {LI_s.mean():.3f} +- {stats.sem(LI_s):.3f} (n={len(LI_s)}); t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: partially reproduced.** The linearity index is 0.941 +- 0.028 for best and 0.723 +- 0.009 for suboptimal combinations (p = 1.5e-16), against 0.93 +- 0.03 and 0.71 +- 0.009: the best mean is within tolerance, the suboptimal mean misses it and lies within 10% of 0.71.

In [ ]:
t = stats.ttest_rel(CSI_m, CSI_l)
print(f"CSI measured {CSI_m.mean():.4f}, linear {CSI_l.mean():.4f}; measured above linear in {np.sum(CSI_m > CSI_l)} of {len(CSI_m)} ({np.mean(CSI_m > CSI_l):.3f}); paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** Measured CSI (0.057) is above linear CSI (0.023) in 70 of 74 units (0.946), paired t-test p = 9e-9.

In [ ]:
za, zb_ = [], []
for _, r in R3.iterrows():
    meas = X3[r.unit]["meas"]
    z_ = (meas - meas.mean()) / meas.std()
    za.append(z_.max(0))
    c = np.unravel_index(np.argmax(meas), meas.shape)[0]
    zb_.append(z_[c])
for lab, arr in [("(a) largest of the eight at each dt", np.array(za)), ("(b) the unit's best combination at each dt", np.array(zb_))]:
    f = stats.f_oneway(*arr.T)
    print(f"{lab}: means at -40, 0, +40 ms {arr.mean(0).round(3)}; ANOVA F = {f.statistic:.2f}, p = {f.pvalue:.3g}")

**Result: reproduced.** Z-scored peak responses are largest at -40 ms under both readings (1.95, 0.93 and 1.36 at -40, 0 and +40 ms, ANOVA p = 8e-14; 1.75, 0.10 and 0.64, p = 2.6e-15).

### fig8k-awake-binary-decoder

> In awake mice, decoding of best against suboptimal CW-SW combinations from single-trial firing was accurate and efficient.

In [ ]:
rng_8k = np.random.default_rng(88)
rows = []
for mode in "ab":
    probs, labs = [], []
    for _, r in R3[R3.sel].iterrows():
        d = X3[r.unit]
        c, j = np.unravel_index(np.argmax(d["meas"]), d["meas"].shape)
        m = d["dti"] == j
        x, yv = d["y"][m], d["comb"][m]
        best = yv == c
        if mode == "a":
            sub = rng_8k.choice(np.where(~best)[0], best.sum(), replace=best.sum() > (~best).sum())
        else:
            sub = np.where(yv == rng_8k.choice([q for q in range(8) if q != c]))[0]
        ii = np.r_[np.where(best)[0], sub]
        lab = np.r_[np.ones(best.sum(), int), np.zeros(len(sub), int)]
        probs.append(L.heldout_prob_rows(x[ii], lab, 2, rng_8k)); labs.append(lab)
    pool = L.Pool(probs, labs, 2)
    r_ = dict(suboptimal=mode, units=int(R3.sel.sum()))
    for N in [5, 10, 15, 20, 25]:
        r_[f"N{N}"] = pool.accuracy(N, 2500, rng_8k).mean()
    rows.append(r_)
print(pd.DataFrame(rows).to_string(index=False, float_format=fmt))

**Result: reproduced.** Over the 27 selective units, accuracy rises with ensemble size, from 0.652 and 0.674 at N = 5 to 0.815 and 0.829 at N = 25 (readings a and b).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| exp1-units | reproduced (assumption: caudal singles) |
| fig1b-somatotopy | reproduced |
| exp1-cw-tuned-fraction | reproduced (assumption: caudal singles) |
| exp1-cw-sw-comparable-firing | partially reproduced |
| supp1d-sw-tuning-broader | reproduced |
| fig1d-best-sequence-exceeds-single | reproduced |
| fig1e-best-combination-includes-cw | reproduced (assumption: same-direction sequences) |
| fig1f-cwsw-exceeds-swsw | reproduced |
| fig1g-cwsw-sharper | reproduced |
| supp2-sw-tuned-sharpest-cwsw | reproduced |
| exp2-units | reproduced |
| methods-dt-modulation | not reproduced |
| exp2-combination-selective-prevalence | not reproduced |
| fig2f-shuffle-abolishes-tuning | reproduced |
| fig2f-most-selective-lower-firing | reproduced |
| supp4-selectivity-at-high-rates | partially reproduced (assumption: the selective group of the shuffle test) |
| fig2g-best-combination-bias | reproduced |
| fig3a-sharper-than-linear | reproduced |
| fig3b-sharper-in-both-groups | reproduced |
| fig3b-linear-sharper-in-selective | reproduced |
| supp6-layers | partially reproduced |
| fig3e-identity-decoder | not reproduced |
| fig3g-binary-decoder | partially reproduced (assumption: the selective group of the shuffle test) |
| fig4a-sw-tuned-sharper | reproduced |
| fig4c-sw-tuned-decoding | partially reproduced |
| fig5a-global-scaling | reproduced |
| fig5bc-linearity-index | partially reproduced |
| fig5c-pseudo-unit-shuffle | reproduced |
| fig5c-sublinear-at-all-dt | reproduced |
| supralinear-best-8-percent | reproduced (assumption: the selective group of the shuffle test) |
| fig6b-best-above-worst-below-scaled | reproduced |
| fig6b-enhancement-not-different | not reproduced (assumption: the selective group of the shuffle test) |
| fig6b-suppression-greater-selective | reproduced |
| fig6c-enhancement-strongest-sw | partially reproduced |
| fig7a-dt-half-width | partially reproduced (assumptions: sets without a best dt left out of the alignment; the selective group of the shuffle test) |
| fig7b-inbound-best-outbound-suboptimal | partially reproduced |
| fig7c-pca | partially reproduced |
| fig7e-pc1-best-suboptimal | partially reproduced |
| fig7f-inbound-sharper | partially reproduced |
| fig7g-inbound-decoding | not reproduced |
| supp8-order-decoding | not reproduced (assumption: the selective group of the shuffle test) |
| exp3-units-cw-tuned | reproduced |
| fig8c-awake-cw-stronger | not reproduced |
| fig8d-awake-best-sequence | reproduced |
| exp3-combination-selective | reproduced |
| fig8h-awake-linearity | partially reproduced |
| fig8i-awake-sharper-than-linear | reproduced |
| fig8j-awake-inbound-stronger | reproduced |
| fig8k-awake-binary-decoder | reproduced |
| fig8b-awake-behavior | blocked: data missing (no lick times or trial types) |

How the analysis differs from the paper's:

- The paper's code is not public; every method comes from the claims file and the Methods. Details the paper leaves open are run under each reading the claims file names.
- Experiment 1: the files hold both deflection directions of each single whisker; the caudal ones are taken as the paper's nine single-whisker stimuli (they give the paper's unit counts; the pooled reading is shown). Sequences use same-direction trials, as the paper's 261 stimuli hold no opposite-direction pairs; pooling the opposite-direction trials of the 20170510 recordings is shown as a variant.
- Experiment 2: one of two units with identical spike trains is dropped (246 cells, 463 responsive sets). Units are grouped by their trial sequence, not by recName. The selective set found by the shuffle test (318 sets) is larger than the paper's (187), and every later claim on selective units uses it. On a real-data null the test gives p < 0.05 for a quarter of the sets, so it is not calibrated here; every selective-group claim is also run on the 187 sets with the smallest p, which changes the outcome of four claims (supp4, fig7a, fig3g, supp8) and of fig6b-enhancement-not-different under one reading, and on the 162 sets that FDR selects from p-values calibrated on that null, which changes the outcome of fig3g, supp8 and supralinear-best-8-percent and of fig6b-enhancement-not-different under one reading. Sets whose best-combination curve is set constant across dt have no defined best dt; they are flagged, their responses are taken over all dt, and for Fig. 7a and reading (c) of Fig. 7f they are left out.
- Experiment 3: lick times are not in the files, so epochs near licks are not excluded; epochs are aligned to the first deflection, as the CW latencies show.

What was found about the dataset:

- The three experiments are present with the paper's counts of units (160, 247 and 81), recording sites (8 in experiment 1), mice (5, 12 and 4) and sessions (6 in experiment 3).
- Experiment 1 files hold rostral as well as caudal single-whisker deflections (18 single-whisker stimuli, not the paper's 9). In both 20170510 recordings about half the trials of each pair involving whisker 3 or 8 have that whisker deflected rostrally and the other caudally, and the pair 3-8 has only opposite-direction trials; the other pairs have only caudal-caudal trials.
- Nine experiment 2 recordings hold 16,000 to 24,000 three-whisker trials, which neither the paper nor the data descriptor mentions. CW-SW dt takes 101 values (-50 to +50 ms), not 100.
- Two experiment 2 units (20160909, recNames chn3 and chn4, depths 250 and 150 um) have identical spike trains. In 20160803 and 20160909 some units carry the recName of another recording than the one whose trials they hold. Two experiment 2 recordings (20160225, 20160316) have no sham trials.
- Experiment 3 files have no deflection times, no lick times, no trial types and no per-unit depth; the DeltaT sign convention is not documented. About a third of experiment 3 spike times lie after the 300 ms epoch, up to 425 s, in 5.6% of unit-epochs.